# 02 — Enrich Court cases and collect documents

This notebook takes the combined Court collection from notebook 01, requests procedural metadata, records joined proceedings, identifies Court documents, and downloads selected source files. The source is InfoCuria.

`affId` routes the procedural request; the returned record is matched to the requested `procedureId` before metadata and documents are assigned. Both identifiers and the published case number are retained.


## Inputs and outputs

Input: `output/court_infocuria/court_cases_discovered.xlsx`.

Outputs, directly under `output/court_infocuria/`:

- `court_cases.xlsx`: enriched collection and document counts.
- `court_documents.xlsx`: document metadata and relevance.
- `court_document_downloads.xlsx`: selected raw files and download status; notebook 03 reads this workbook.

Raw HTML/PDF files stay in `data/raw/court_infocuria/`. Restartable metadata, joined-case, and download state stays as JSONL in `logs/court_infocuria/`.


In [1]:
# ---------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------
import json
import os
import re
import time
import asyncio
import logging
import random
from pathlib import Path
from datetime import datetime
from collections import Counter
from typing import Any
from urllib.parse import urlparse, quote

import pandas as pd
import numpy as np
import requests
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("enrich_infocuria_document_layer")


## 1. Paths, document types, and acquisition settings

The workbook paths below are the handoffs between notebooks. JSONL files under `logs/court_infocuria/` allow long metadata requests and downloads to resume after interruption. They are working state, rather than additional dataset tables.


In [2]:
# Repository paths

def find_project_root(start: Path | None = None) -> Path:
    override = os.environ.get("ECCJEU_PROJECT_ROOT", "").strip()
    if override:
        root = Path(override).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(f"ECCJEU_PROJECT_ROOT does not exist: {root}")
        return root
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "code" / "02_court").exists():
            return candidate
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the ec_cjeu project root. Run the notebook from within "
        "the repository or set ECCJEU_PROJECT_ROOT."
    )

PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / "output" / "court_infocuria"
LOG_DIR = PROJECT_ROOT / "logs" / "court_infocuria"
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "court_infocuria"
HTML_DIR = RAW_DIR / "html"
PDF_DIR = RAW_DIR / "pdf"

for directory in [OUTPUT_DIR, LOG_DIR, HTML_DIR, PDF_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

COURT_CASE_INPUT = OUTPUT_DIR / "court_cases_discovered.xlsx"
if not COURT_CASE_INPUT.exists():
    raise FileNotFoundError(f"Run notebook 01 first: {COURT_CASE_INPUT}")

CASE_METADATA_XLSX = OUTPUT_DIR / "court_cases.xlsx"
DOCUMENT_METADATA_XLSX = OUTPUT_DIR / "court_documents.xlsx"
DOWNLOAD_MANIFEST_XLSX = OUTPUT_DIR / "court_document_downloads.xlsx"

# Resumable acquisition state. JSONL is kept in logs only.
RAW_METADATA_JSONL = LOG_DIR / "procedural_metadata_state.jsonl"
JOINED_CASE_JSONL = LOG_DIR / "joined_case_state.jsonl"
DOWNLOAD_MANIFEST_JSONL = LOG_DIR / "document_download_state.jsonl"
FLATTEN_FAILED_JSONL = LOG_DIR / "metadata_flatten_failures.jsonl"

# Metadata acquisition mode
SCRAPE_MODE = "update_scrape"
VALID_SCRAPE_MODES = {"fresh_scrape", "update_scrape", "rebuild_from_cache"}
if SCRAPE_MODE not in VALID_SCRAPE_MODES:
    raise ValueError(f"Unknown SCRAPE_MODE: {SCRAPE_MODE}")
LIMIT_METADATA_FETCH_CASES = None
UPDATE_EXISTING_CACHE_RECORDS = False

PROCEDURES_URL = "https://infocuriaws.curia.europa.eu/elastic-connector/affairId/procedures"
LANGUAGE = "EN"
METADATA_REQUEST_TIMEOUT_SECONDS = 60
METADATA_MAX_RETRIES = 5
METADATA_SLEEP_SECONDS = 0.25

RELEVANT_DOC_TYPE_CODES = {
    "ARRET", "ARRET_NP", "ARRET_DR", "ARRET_SOM",
    "ORD", "ORD_NP", "ORD_DR", "ORD_SOM", "ORD_COMM",
    "RECT", "DECISION",
}
RELEVANCE_REASON = {
    "ARRET": "relevant_final_judgment",
    "ARRET_NP": "relevant_final_judgment_not_published",
    "ARRET_DR": "relevant_judgment_dr_variant",
    "ARRET_SOM": "relevant_summary_judgment_variant",
    "ORD": "relevant_final_order",
    "ORD_NP": "relevant_final_order_not_published",
    "ORD_DR": "relevant_order_dr_variant",
    "ORD_SOM": "relevant_summary_order_variant",
    "ORD_COMM": "relevant_order_communication_variant",
    "RECT": "relevant_rectification_corrigendum",
    "DECISION": "relevant_decision",
}

RUN_DOWNLOADS = True
OVERWRITE_EXISTING_FILES = False
DOWNLOAD_LIMIT = None
DOWNLOAD_SAVE_EVERY = 100
PREFERRED_LANGUAGES = ["EN", "DE", "FR"]
PREFERRED_FORMATS = ["HTML", "PDF"]
REQUEST_TIMEOUT_SECONDS = 90
DOWNLOAD_SLEEP_SECONDS = 1.25
DOWNLOAD_SLEEP_JITTER_SECONDS = 0.75
RETRIES_PER_URL = 3
RETRY_BACKOFF_SECONDS = 4.0
SESSION_WARMUP = True
SESSION_WARMUP_SLEEP_SECONDS = 0.75
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/126 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml,application/pdf,application/json;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9,de;q=0.8,fr;q=0.7",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Origin": "https://infocuria.curia.europa.eu",
    "Referer": "https://infocuria.curia.europa.eu/",
}

STREAM_JSONL_CHUNK_SIZE = 5000
KEEP_RAW_CONTENT_JSON = False
DROP_NON_ENGLISH_METADATA_COLUMNS = [
    "case_name_de", "case_name_fr", "case_object_de", "case_object_fr",
    "subject_matter_de", "subject_matter_fr", "procedure_result_de", "procedure_result_fr",
]
DROP_HUGE_DOWNLOAD_COLUMNS = ["candidate_download_records", "all_candidate_download_urls"]

# Joined-proceeding repair from the InfoCuria procedures endpoint.
RUN_JOINED_CASE_SEARCH = True
JOINED_CASE_SEARCH_LIMIT = None
JOINED_CASE_SLEEP_SECONDS = 0.15
JOINED_CASE_JSONL_MODE = "update_failed"
VALID_JOINED_CASE_JSONL_MODES = {"rebuild_jsonl", "update_failed", "csv_from_jsonl"}
if JOINED_CASE_JSONL_MODE not in VALID_JOINED_CASE_JSONL_MODES:
    raise ValueError(f"Unknown JOINED_CASE_JSONL_MODE: {JOINED_CASE_JSONL_MODE}")
JOINED_CASE_RETRY_HTTP_STATUSES = {403, 429, 500, 502, 503, 504}
JOINED_CASE_BATCH_PAUSE_EVERY = 1000
JOINED_CASE_BATCH_PAUSE_SECONDS = 60
JOINED_CASE_COMPACT_JSONL_AFTER_RUN = True
JOINED_CASE_RECORD_RESPONSE_SUMMARY = True

COURT_INFOCURIA_RAW_DIR = RAW_DIR
COURT_INFOCURIA_HTML_DIR = HTML_DIR
COURT_INFOCURIA_PDFS_DIR = PDF_DIR
COURT_INFOCURIA_HTML_EXISTING_DIRS = [HTML_DIR]
COURT_INFOCURIA_PDF_EXISTING_DIRS = [PDF_DIR]

CASE_METADATA_STEM = "court_cases"
DOCUMENT_METADATA_STEM = "court_documents"
DOWNLOAD_MANIFEST_STEM = "court_document_downloads"


def read_manifest(path: Path) -> pd.DataFrame:
    path = Path(path)
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path, dtype=str)
    if path.suffix.lower() == ".jsonl":
        return pd.read_json(path, orient="records", lines=True, dtype=False)
    return pd.read_csv(path, dtype=str, low_memory=False)


def save_jsonl(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_json(path, orient="records", lines=True, force_ascii=False)


def save_table(df: pd.DataFrame, xlsx_path: Path, sheet_name: str) -> None:
    df.to_excel(xlsx_path, index=False, sheet_name=sheet_name[:31])


def file_size_mb(path: Path) -> float:
    return path.stat().st_size / (1024 * 1024)


def count_jsonl_lines(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open("rb") as handle:
        return sum(1 for _ in handle)

logger = logging.getLogger("court_infocuria_documents")
logger.setLevel(logging.INFO)
if not logger.handlers:
    logger.addHandler(logging.StreamHandler())

print("Project root:", PROJECT_ROOT)
print("Case manifest:", COURT_CASE_INPUT)
print("Procedural metadata state:", RAW_METADATA_JSONL)
print("Download state:", DOWNLOAD_MANIFEST_JSONL)


Project root: /home/edik/projects/ec_cjeu
Case manifest: /home/edik/projects/ec_cjeu/output/court_infocuria/court_cases_discovered.xlsx
Procedural metadata state: /home/edik/projects/ec_cjeu/logs/court_infocuria/procedural_metadata_state.jsonl
Download state: /home/edik/projects/ec_cjeu/logs/court_infocuria/document_download_state.jsonl


## 2. Read the discovered cases

Preserve source `affId` and `procedureId` alongside the published case number. The readable normalized keys help reconcile representations, but they do not replace those identifiers. Check the incoming workbook before requesting metadata.


In [3]:
df_cases_in = read_manifest(COURT_CASE_INPUT)
if df_cases_in.empty:
    raise ValueError("The InfoCuria case manifest is empty.")
required = {"procedureId", "affId", "case_number_raw", "internal_key"}
missing = sorted(required - set(df_cases_in.columns))
if missing:
    raise ValueError(f"Case manifest is missing required columns: {missing}")
print(f"Case procedures available: {len(df_cases_in):,}")
display(df_cases_in.head(5))

# ---------------------------------------------------------------------
# Generic helpers
# ---------------------------------------------------------------------

def clean_str(x):
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x).strip()



def first_nonempty(*values):
    """Return the first value that is not None/NA/empty after clean_str().

    This avoids pandas pd.NA boolean errors from expressions like
    `row.get("a") or row.get("b")`.
    """
    for value in values:
        s = clean_str(value)
        if s and s.lower() not in {"nan", "none", "<na>"}:
            return s
    return ""

def path_exists_safe(*values):
    """NA-safe check for the first non-empty path-like value."""
    s = first_nonempty(*values)
    if not s:
        return False
    try:
        return Path(s).exists()
    except Exception:
        return False

def normalize_whitespace(x):
    return re.sub(r"\s+", " ", clean_str(x)).strip()

def as_json_string(x):
    try:
        return json.dumps(x, ensure_ascii=False)
    except Exception:
        return "[]"

def unique_join(values, sep="|", max_items=None):
    vals = []
    for v in values:
        s = clean_str(v)
        if not s or s.lower() in {"nan", "none", "<na>"}:
            continue
        vals.append(s)
    vals = sorted(set(vals))
    if max_items is not None:
        vals = vals[:max_items]
    return sep.join(vals)

def ml_value(obj, lang="en", fallback=True):
    """Extract multilingual value from InfoCuria ML structures."""
    if isinstance(obj, str):
        try:
            obj = json.loads(obj)
        except Exception:
            return obj if fallback else ""
    lang = str(lang).lower()
    if isinstance(obj, list):
        found = []
        for item in obj:
            if isinstance(item, dict):
                for k, v in item.items():
                    if str(k).lower() == lang and clean_str(v):
                        return clean_str(v)
                    if clean_str(v):
                        found.append(clean_str(v))
        return found[0] if fallback and found else ""
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).lower() == lang and clean_str(v):
                return clean_str(v)
        vals = [clean_str(v) for v in obj.values() if clean_str(v)]
        return vals[0] if fallback and vals else ""
    return ""

def label_text(obj, lang="en"):
    """Extract label text from ML label structures."""
    if isinstance(obj, str):
        try:
            obj = json.loads(obj)
        except Exception:
            return obj
    if isinstance(obj, list):
        vals = []
        for item in obj:
            if isinstance(item, dict):
                label = item.get("label", item)
                val = ml_value(label, lang, fallback=True)
                if val:
                    vals.append(val)
        return "|".join(sorted(set(vals)))
    return ml_value(obj, lang, fallback=True)

def parse_affair_ref_list(values):
    """Pull published case IDs from InfoCuria affair reference fields."""
    if values is None:
        return []
    if isinstance(values, str):
        try:
            values = json.loads(values)
        except Exception:
            values = [values]
    if not isinstance(values, list):
        values = [values]
    out = []
    for v in values:
        if isinstance(v, dict):
            for key in ["publishedId", "publishedAffId", "affaire", "label", "value", "id"]:
                if clean_str(v.get(key)):
                    out.append(clean_str(v.get(key)))
                    break
        else:
            s = clean_str(v)
            # Often fields are like "C-123/20".
            m = re.search(r"\b[CT]\s*-?\s*\d{1,5}\s*/\s*\d{2,4}(?:\s+[A-Z][A-Z0-9-]*)?\b", s, flags=re.I)
            if m:
                out.append(normalize_whitespace(m.group(0)))
    return sorted(set(out))

def normalize_doc_type(value):
    return normalize_whitespace(value).upper().replace(" ", "_").replace("-", "_")

def split_doc_type_codes(value):
    """Return normalized document-type tokens.

    InfoCuria sometimes returns combined codes such as ORD_COMM|REQ_COMM.
    Relevance is evaluated token-wise, so ORD_COMM is recognised as
    relevant inside a combined code.
    """
    s = normalize_doc_type(value)
    if not s:
        return []
    return [p for p in re.split(r"[|,;/]+", s) if p]

def relevant_doc_type_tokens(code):
    return [t for t in split_doc_type_codes(code) if t in RELEVANT_DOC_TYPE_CODES]

def is_relevant_doc_type(code):
    return bool(relevant_doc_type_tokens(code))

def relevance_reason(code):
    hits = relevant_doc_type_tokens(code)
    if hits:
        return "|".join(RELEVANCE_REASON.get(h, f"relevant_{h.lower()}") for h in hits)
    return "irrelevant_non_final_or_duplicate_or_administrative_document"


# ---------------------------------------------------------------------
# Case-key helpers
# ---------------------------------------------------------------------

# - The upstream web/case manifest already contains `internal_key`.
# - Use that as the procedural key whenever present.
# - Only parse/generate keys as a fallback for older manifests.

def normalize_case_suffix(suffix):
    suffix = normalize_whitespace(suffix).upper()
    suffix = suffix.replace("–", "-").replace("—", "-")
    suffix = re.sub(r"[^A-Z0-9 -]+", "", suffix)
    suffix = re.sub(r"\s+", "-", suffix).strip("-")
    return suffix

def normalize_internal_key(value):
    """Normalize procedural key to YYYY/NNNN/PREFIX/SUFFIX/.

    Empty suffix is preserved as an empty fourth segment, e.g. 1966/0001/C//.
    This matches the upstream case skeleton internal_key format and prevents
    case/document join mismatches caused by 1966/0001/C/ vs 1966/0001/C//.
    """
    s = clean_str(value).upper().replace("\\", "/")
    if not s:
        return ""
    parts = s.split("/")
    # Preserve an explicitly empty suffix if present; remove only surrounding empty parts.
    if len(parts) >= 3 and parts[0] == "":
        parts = parts[1:]
    if len(parts) >= 4:
        year, num, prefix = parts[0], parts[1], parts[2]
        suffix = parts[3]
    else:
        compact = re.sub(r"/+", "/", s).strip("/")
        parts = compact.split("/") if compact else []
        if len(parts) >= 3:
            year, num, prefix = parts[0], parts[1], parts[2]
            suffix = parts[3] if len(parts) >= 4 else ""
        else:
            return re.sub(r"/+", "/", s if s.endswith("/") else s + "/")
    year = f"{int(year):04d}" if str(year).isdigit() else str(year)
    num = f"{int(num):04d}" if str(num).isdigit() else str(num)
    prefix = clean_str(prefix).upper()
    suffix = normalize_case_suffix(suffix)
    return f"{year}/{num}/{prefix}/{suffix}/"


def internal_key_base(value):
    s = normalize_internal_key(value)
    parts = [p for p in s.split("/") if p]
    if len(parts) >= 3:
        return f"{parts[0]}/{parts[1]}/{parts[2]}/"
    return ""

def make_slash_procedure_keys(prefix, num, year_full, suffix=""):
    prefix = clean_str(prefix).upper()
    suffix = normalize_case_suffix(suffix)
    if not prefix or num is None or not clean_str(year_full):
        return "", ""
    base = f"{int(year_full):04d}/{int(num):04d}/{prefix}/"
    full = f"{int(year_full):04d}/{int(num):04d}/{prefix}/{suffix}/"
    return full, base

def parse_case_number(raw: Any, clean: Any = None, prefix_hint: Any = None, year_hint: Any = None):
    raw_s = normalize_whitespace(raw).upper().replace("–", "-").replace("—", "-")
    clean_s = normalize_whitespace(clean).upper().replace("–", "-").replace("—", "-")
    prefix_hint = clean_str(prefix_hint).upper()

    slash_re = r"\b(\d{4})\s*/\s*(\d{1,5})\s*/\s*([CT])\s*/?\s*([A-Z0-9 -]*)\b"
    m0 = re.search(slash_re, raw_s) or (re.search(slash_re, clean_s) if clean_s else None)
    if m0:
        year_full = int(m0.group(1))
        num = int(m0.group(2))
        prefix = m0.group(3)
        suffix = normalize_case_suffix(m0.group(4))
        procedure_key, procedure_key_base = make_slash_procedure_keys(prefix, num, year_full, suffix)
        return {
            "case_prefix": prefix,
            "case_number_int": num,
            "case_number_int_full": f"{num:04d}",
            "case_number_padded": f"{num:04d}",
            "case_year_raw": str(year_full),
            "case_year_full": str(year_full),
            "case_suffix": suffix,
            "procedure_key": procedure_key,
            "procedure_key_base": procedure_key_base,
        }

    m = re.search(r"\b([CT])\s*-?\s*(\d{1,5})\s*/\s*(\d{2,4})\b\s*(.*)$", raw_s)
    if not m and clean_s:
        m = re.search(r"\b([CT])\s*-?\s*(\d{1,5})\s*/\s*(\d{2,4})\b\s*(.*)$", clean_s)

    if m:
        prefix = m.group(1)
        num = int(m.group(2))
        year_raw = m.group(3)
        suffix = normalize_case_suffix(m.group(4))
    else:
        m2 = re.search(r"\b(\d{1,5})\s*/\s*(\d{2,4})\b\s*(.*)$", raw_s or clean_s)
        if not m2:
            return {
                "case_prefix": prefix_hint or "",
                "case_number_int": None,
                "case_number_int_full": "",
                "case_number_padded": "",
                "case_year_raw": clean_str(year_hint),
                "case_year_full": "",
                "case_suffix": "",
                "procedure_key": "",
                "procedure_key_base": "",
            }
        prefix = prefix_hint or "C"
        num = int(m2.group(1))
        year_raw = m2.group(2)
        suffix = normalize_case_suffix(m2.group(3))

    year_int = int(year_raw)
    year_full = 1900 + year_int if year_int < 100 and year_int >= 50 else 2000 + year_int if year_int < 100 else year_int
    procedure_key, procedure_key_base = make_slash_procedure_keys(prefix, num, year_full, suffix)
    return {
        "case_prefix": prefix,
        "case_number_int": num,
        "case_number_int_full": f"{num:04d}",
        "case_number_padded": f"{num:04d}",
        "case_year_raw": year_raw,
        "case_year_full": str(year_full),
        "case_suffix": suffix,
        "procedure_key": procedure_key,
        "procedure_key_base": procedure_key_base,
    }

def add_or_preserve_procedure_key_columns(df, raw_col=None, clean_col=None, prefix_col=None, year_col=None, prefer_internal_key=True):
    if df.empty:
        return df.copy()
    out = df.copy()

    internal_col = next((c for c in ["internal_key", "procedure_key"] if c in out.columns), None) if prefer_internal_key else None
    has_internal = internal_col is not None and out[internal_col].fillna("").astype(str).str.strip().ne("")

    parsed_rows = []
    if raw_col is None:
        raw_col = next((c for c in ["case_number_raw", "publishedId", "case_number", "case", "affaire", "procedure_key"] if c in out.columns), None)
    if clean_col is None:
        clean_col = next((c for c in ["case_number_clean", "case_number", "procedure_key_base"] if c in out.columns), None)
    if prefix_col is None:
        prefix_col = next((c for c in ["case_prefix", "prefix"] if c in out.columns), None)
    if year_col is None:
        year_col = next((c for c in ["case_year_full", "year"] if c in out.columns), None)

    for _, row in out.iterrows():
        parsed_rows.append(parse_case_number(
            row.get(raw_col) if raw_col else "",
            row.get(clean_col) if clean_col else "",
            row.get(prefix_col) if prefix_col else "",
            row.get(year_col) if year_col else "",
        ))
    parsed_df = pd.DataFrame(parsed_rows, index=out.index)

    # Fill missing key component columns, but do not overwrite an upstream internal_key.
    for col in ["case_prefix", "case_number_int", "case_number_int_full", "case_number_padded", "case_year_raw", "case_year_full", "case_suffix"]:
        if col not in out.columns:
            out[col] = parsed_df.get(col)
        else:
            out[col] = out[col].where(out[col].fillna("").astype(str).str.strip().ne(""), parsed_df.get(col))

    if internal_col and prefer_internal_key:
        out["procedure_key"] = out[internal_col].map(normalize_internal_key)
        out["procedure_key_base"] = out["procedure_key"].map(internal_key_base)
    else:
        out["procedure_key"] = parsed_df["procedure_key"]
        out["procedure_key_base"] = parsed_df["procedure_key_base"]

    # Keep `internal_key` explicit in outputs.
    if "internal_key" not in out.columns:
        out["internal_key"] = out["procedure_key"]

    return out

df_cases = add_or_preserve_procedure_key_columns(
    df_cases_in,
    raw_col="case_number_raw" if "case_number_raw" in df_cases_in.columns else None,
    prefer_internal_key=True,
)

print("InfoCuria cases keyed:", df_cases.shape)
if not df_cases.empty and "procedure_key" in df_cases.columns:
    display(df_cases[[c for c in ["internal_key", "case_number_raw", "case_number_clean", "publishedId", "affId", "procedureId", "procedure_key", "procedure_key_base"] if c in df_cases.columns]].head(10))


Case procedures available: 57,413


,case_number_raw,case_number_clean,case_prefix,case_suffix,case_number_int,case_number_padded,case_year_raw,case_year_full,internal_key,affId,publishedId,publishedAffId,procedureId,case_name_en,jurisdictionCode,natureCode,introductionDate,closeDate,affairStateCode,subject_matter_text,procLang,ecli,joinExist,joinAffairs
0,C-1/00,C-1/00,C,NaN,1,0001,00,2000,2000/0001/C//,C/0001/00/00000000RD/01,C-1/00,C-1/00,C/0001/00/00000000RD/01/P/01,Commission v France,C,NaN,2000-01-04,2001-12-13,CLOTPUB,NaN,NaN,NaN,0,[]
1,C-1/00 SA,C-1/00 SA,C,SA,1,0001,00,2000,2000/0001/C/SA/,C/0001/00/00000000SA/01,C-1/00 SA,C-1/00 SA,C/0001/00/00000000SA/01/P/01,Cotecna Inspection v Commission,C,NaN,2000-12-14,2001-05-29,CLOTPUB,NaN,NaN,NaN,0,[]
2,C-1/01 P,C-1/01 P,C,P,1,0001,01,2001,2001/0001/C/P/,C/0001/01/00000000PV/01,C-1/01 P,C-1/01 P,C/0001/01/00000000PV/01/P/01,Asia Motor France and Others v Commission,C,NaN,2001-01-03,2001-09-20,CLOTPUB,NaN,NaN,NaN,0,[]
3,C-1/02,C-1/02,C,NaN,1,0001,02,2002,2002/0001/C//,C/0001/02/00000000RP/01,C-1/02,C-1/02,C/0001/02/00000000RP/01/P/01,Borgmann,C,NaN,2002-01-07,2004-04-01,CLOTPUB,NaN,NaN,NaN,0,[]
4,C-1/02 SA,C-1/02 SA,C,SA,1,0001,02,2002,2002/0001/C/SA/,C/0001/02/00000000SA/01,C-1/02 SA,C-1/02 SA,C/0001/02/00000000SA/01/P/01,Antippas v Commission,C,NaN,2002-08-22,2003-03-27,CLOTPUB,NaN,NaN,NaN,0,[]


InfoCuria cases keyed: (57413, 27)


,internal_key,case_number_raw,case_number_clean,publishedId,affId,procedureId,procedure_key,procedure_key_base
0,2000/0001/C//,C-1/00,C-1/00,C-1/00,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,2000/0001/C//,2000/0001/C/
1,2000/0001/C/SA/,C-1/00 SA,C-1/00 SA,C-1/00 SA,C/0001/00/00000000SA/01,C/0001/00/00000000SA/01/P/01,2000/0001/C/SA/,2000/0001/C/
2,2001/0001/C/P/,C-1/01 P,C-1/01 P,C-1/01 P,C/0001/01/00000000PV/01,C/0001/01/00000000PV/01/P/01,2001/0001/C/P/,2001/0001/C/
3,2002/0001/C//,C-1/02,C-1/02,C-1/02,C/0001/02/00000000RP/01,C/0001/02/00000000RP/01/P/01,2002/0001/C//,2002/0001/C/
4,2002/0001/C/SA/,C-1/02 SA,C-1/02 SA,C-1/02 SA,C/0001/02/00000000SA/01,C/0001/02/00000000SA/01/P/01,2002/0001/C/SA/,2002/0001/C/
5,2003/0001/C//,C-1/03,C-1/03,C-1/03,C/0001/03/00000000RP/01,C/0001/03/00000000RP/01/P/01,2003/0001/C//,2003/0001/C/
6,2003/0001/C/SA/,C-1/03 SA,C-1/03 SA,C-1/03 SA,C/0001/03/00000000SA/01,C/0001/03/00000000SA/01/P/01,2003/0001/C/SA/,2003/0001/C/
7,2004/0001/C//,C-1/04,C-1/04,C-1/04,C/0001/04/00000000RP/01,C/0001/04/00000000RP/01/P/01,2004/0001/C//,2004/0001/C/
8,2004/0001/C/SA/,C-1/04 SA,C-1/04 SA,C-1/04 SA,C/0001/04/00000000SA/01,C/0001/04/00000000SA/01/P/01,2004/0001/C/SA/,2004/0001/C/
9,2005/0001/C//,C-1/05,C-1/05,C-1/05,C/0001/05/00000000RP/01,C/0001/05/00000000RP/01/P/01,2005/0001/C//,2005/0001/C/


## 3. Retrieve procedural metadata

The endpoint accepts `affId` for the request. Because a response can contain more than one procedural record, match its result back to the requested `procedureId` before attaching metadata or documents. Failed requests remain in JSONL state for a later retry.


In [ ]:
# ---------------------------------------------------------------------
# Optional affairId/procedures metadata fetch/cache construction
# ---------------------------------------------------------------------

def metadata_post_json(url, payload, max_retries=METADATA_MAX_RETRIES):
    last_error = None
    metadata_headers = {
        **HEADERS,
        "accept": "application/json",
        "content-type": "application/json; charset=UTF-8",
    }
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(
                url,
                headers=metadata_headers,
                json=payload,
                timeout=METADATA_REQUEST_TIMEOUT_SECONDS,
            )
            if response.status_code >= 400:
                last_error = f"HTTP {response.status_code}: {response.text[:500]}"
                raise RuntimeError(last_error)
            return response.json()
        except Exception as e:
            last_error = repr(e)
            wait = min(60, 2 ** attempt) + random.uniform(0, 1.0)
            logger.warning("Metadata request failed attempt %s/%s: %s | sleeping %.1fs", attempt, max_retries, last_error, wait)
            time.sleep(wait)
    raise RuntimeError(f"Metadata request failed after {max_retries} retries: {last_error}")



def fetch_procedure_record(
    aff_id: str,
    published_id: str = "",
    language: str = LANGUAGE,
):
    """
    Query the affairId/procedures endpoint for one procedure-manifest row.

    The endpoint is routed by affId, while the exact procedure label is passed
    in searchTerm. The returned response is later filtered to the requested
    procedureId before case metadata and documents are flattened.
    """
    payload = {
        "affId": aff_id,
        "searchTerm": f'"{published_id}"' if published_id else "",
        "tabName": "affair",
        "language": language,
        "advancedFiltersValue": [],
    }
    return metadata_post_json(PROCEDURES_URL, payload)


def append_raw_jsonl(path: Path, record: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8", newline="\n") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def iter_raw_jsonl(path: Path):
    if not path.exists():
        return
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except Exception as e:
                logger.warning("Skipping bad JSONL line %s: %r", line_no, e)


def raw_jsonl_key(record: dict) -> str:
    """
    procedureId is the canonical cache key.

    Fallbacks are retained only for rows where procedureId
    is absent.
    """
    return first_nonempty(
        record.get("procedureId"),
        record.get("internal_key"),
        record.get("procedure_key"),
        record.get("publishedId"),
        record.get("affId"),
    )


def compact_raw_jsonl_keep_last(path: Path):
    """Rewrite the JSONL so each procedure key appears once, keeping the last record."""
    if not path.exists():
        return

    latest = {}
    order = []

    for rec in iter_raw_jsonl(path):
        key = raw_jsonl_key(rec)
        if not key:
            key = f"__no_key__{len(order)}"

        if key not in latest:
            order.append(key)

        latest[key] = rec

    tmp = path.with_suffix(path.suffix + ".tmp")

    with open(tmp, "w", encoding="utf-8", newline="\n") as f:
        for key in order:
            f.write(json.dumps(latest[key], ensure_ascii=False) + "\n")

    tmp.replace(path)
    print(
        f"Compacted raw JSONL to {len(latest):,} unique procedure records "
        f"-> {path}"
    )


def load_raw_jsonl_keys(path: Path) -> set:
    keys = set()

    if not path.exists():
        return keys

    for obj in tqdm(
        iter_raw_jsonl(path),
        desc="Scanning procedure-level raw JSONL keys",
    ):
        key = raw_jsonl_key(obj)
        if key:
            keys.add(key)

    return keys


def prepare_procedure_fetch_spine(
    df_cases_source: pd.DataFrame,
) -> pd.DataFrame:
    """
    Produce one input row per procedureId.

    Rows with a valid procedureId are deduplicated only by procedureId.
    A fallback path is retained for rows missing procedureId so they can be
    logged and inspected rather than silently lost.
    """
    df_fetch = df_cases_source.copy()

    if "procedureId" not in df_fetch.columns:
        raise KeyError(
            "The InfoCuria case manifest must contain a procedureId column."
        )

    df_fetch["procedureId"] = (
        df_fetch["procedureId"]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

    has_procedure_id = df_fetch["procedureId"].notna()

    with_procedure_id = (
        df_fetch.loc[has_procedure_id]
        .drop_duplicates(subset=["procedureId"], keep="last")
    )

    without_procedure_id = df_fetch.loc[~has_procedure_id].copy()

    fallback_cols = [
        c
        for c in ["affId", "internal_key", "publishedId"]
        if c in without_procedure_id.columns
    ]

    if fallback_cols and not without_procedure_id.empty:
        without_procedure_id = without_procedure_id.drop_duplicates(
            subset=fallback_cols,
            keep="last",
        )

    df_fetch = pd.concat(
        [with_procedure_id, without_procedure_id],
        ignore_index=True,
    ).reset_index(drop=True)

    print(
        "Procedure-level fetch spine:",
        f"{len(df_fetch):,} rows |",
        f"{df_fetch['procedureId'].nunique(dropna=True):,} distinct procedureIds |",
        f"{df_fetch['procedureId'].isna().sum():,} rows missing procedureId",
    )

    return df_fetch


def build_or_update_metadata_jsonl(
    df_cases_source: pd.DataFrame,
):
    mode = SCRAPE_MODE

    if mode == "rebuild_from_cache":
        print(
            "SCRAPE_MODE='rebuild_from_cache': using existing "
            "procedure-level JSONL checkpoint only:",
            RAW_METADATA_JSONL,
        )
        return

    if mode == "fresh_scrape" and RAW_METADATA_JSONL.exists():
        RAW_METADATA_JSONL.unlink()
        print(
            "Deleted old procedure-level raw JSONL for fresh scrape:",
            RAW_METADATA_JSONL,
        )

    df_fetch = prepare_procedure_fetch_spine(df_cases_source)

    if LIMIT_METADATA_FETCH_CASES is not None:
        df_fetch = df_fetch.head(
            int(LIMIT_METADATA_FETCH_CASES)
        ).copy()

    raw_cache_keys = (
        load_raw_jsonl_keys(RAW_METADATA_JSONL)
        if RAW_METADATA_JSONL.exists()
        else set()
    )

    run_log_rows = []
    failed_rows = []

    for _, row in tqdm(
        df_fetch.iterrows(),
        total=len(df_fetch),
        desc=(
            f"{mode}: fetching InfoCuria procedure-level "
            "affairId/procedures metadata"
        ),
    ):
        procedure_id = clean_str(row.get("procedureId"))
        aff_id = clean_str(row.get("affId"))
        internal_key = clean_str(
            row.get("internal_key")
            or row.get("procedure_key")
        )
        published_id = clean_str(
            row.get("publishedId")
            or row.get("case_number_raw")
            or row.get("case_number_clean")
        )

        cache_key = first_nonempty(
            procedure_id,
            internal_key,
            published_id,
            aff_id,
        )

        if not procedure_id:
            failed_rows.append({
                "affId": aff_id,
                "procedureId": procedure_id,
                "internal_key": internal_key,
                "publishedId": published_id,
                "procedure_key": row.get("procedure_key"),
                "error": "missing_procedureId",
            })
            continue

        if not aff_id:
            failed_rows.append({
                "affId": aff_id,
                "procedureId": procedure_id,
                "internal_key": internal_key,
                "publishedId": published_id,
                "procedure_key": row.get("procedure_key"),
                "error": "missing_affId",
            })
            continue

        if (
            mode == "update_scrape"
            and cache_key in raw_cache_keys
            and not UPDATE_EXISTING_CACHE_RECORDS
        ):
            run_log_rows.append({
                "affId": aff_id,
                "procedureId": procedure_id,
                "internal_key": internal_key,
                "publishedId": published_id,
                "procedure_key": row.get("procedure_key"),
                "status": "cached_skipped",
            })
            continue

        try:
            data = fetch_procedure_record(
                aff_id=aff_id,
                published_id=published_id,
                language=LANGUAGE,
            )

            rec = {
                "fetched_at": (
                    datetime.utcnow()
                    .isoformat(timespec="seconds")
                    + "Z"
                ),
                "cache_key": cache_key,
                "affId": aff_id,
                "procedureId": procedure_id,
                "internal_key": internal_key,
                "publishedId": published_id,
                "procedure_key": row.get("procedure_key"),
                "response": data,
            }

            append_raw_jsonl(
                RAW_METADATA_JSONL,
                rec,
            )

            raw_cache_keys.add(cache_key)

            run_log_rows.append({
                "affId": aff_id,
                "procedureId": procedure_id,
                "internal_key": internal_key,
                "publishedId": published_id,
                "procedure_key": row.get("procedure_key"),
                "status": "ok",
                "totalHits": data.get("totalHits"),
            })

        except Exception as e:
            failed_rows.append({
                "affId": aff_id,
                "procedureId": procedure_id,
                "internal_key": internal_key,
                "publishedId": published_id,
                "procedure_key": row.get("procedure_key"),
                "error": repr(e),
            })

            run_log_rows.append({
                "affId": aff_id,
                "procedureId": procedure_id,
                "internal_key": internal_key,
                "publishedId": published_id,
                "procedure_key": row.get("procedure_key"),
                "status": "failed",
                "error": repr(e),
            })

        time.sleep(float(METADATA_SLEEP_SECONDS))

    df_log = pd.DataFrame(run_log_rows)
    df_fail = pd.DataFrame(failed_rows)

    compact_raw_jsonl_keep_last(
        RAW_METADATA_JSONL
    )

    print("Procedure-level metadata fetch/update pass complete.")
    print("Fetch log rows:", f"{len(run_log_rows):,}")
    print("Failures:", f"{len(failed_rows):,}")


build_or_update_metadata_jsonl(df_cases)

if not RAW_METADATA_JSONL.exists():
    raise FileNotFoundError(f"Raw metadata JSONL still does not exist: {RAW_METADATA_JSONL}")

print("Raw JSONL ready:", RAW_METADATA_JSONL)
print("Raw JSONL size MB:", round(file_size_mb(RAW_METADATA_JSONL), 2))
print("Raw JSONL line count:", f"{count_jsonl_lines(RAW_METADATA_JSONL):,}")


Procedure-level fetch spine: 57,413 rows | 57,413 distinct procedureIds | 0 rows missing procedureId


update_scrape: fetching InfoCuria procedure-level affairId/procedures metadata:   0%|          | 0/57413 [00:0…

/tmp/ipykernel_3222126/3555852712.py:308: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow()


## 4. Construct case and document rows

Flatten the selected procedural record and its documents. Keep the source identifiers on both tables. Document-type codes define which documents enter the downloadable collection; cases without a selected document remain in the case collection.


In [ ]:
# ---------------------------------------------------------------------
# Raw JSONL iterator and InfoCuria flattening helpers
# ---------------------------------------------------------------------

def iter_raw_jsonl(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except Exception as e:
                logger.warning("Skipping bad JSONL line %s: %r", line_no, e)

def select_requested_procedure_hit(raw_rec):
    """
    Select the response hit matching the requested procedureId.

    Matching order:
      1. exact procedureId
      2. exact publishedId
      3. exact affId only when the response contains one hit
      4. no match -> return an empty result rather than silently using
         the wrong procedure
    """
    response = raw_rec.get("response") or {}
    hits = response.get("searchHits") or []

    if not hits:
        return None, {}

    requested_procedure_id = clean_str(
        raw_rec.get("procedureId")
    )

    if requested_procedure_id:
        for hit in hits:
            content = hit.get("content") or {}

            if clean_str(
                content.get("procedureId")
            ) == requested_procedure_id:
                return hit, content

    requested_published_id = clean_str(
        raw_rec.get("publishedId")
    )

    if requested_published_id:
        for hit in hits:
            content = hit.get("content") or {}

            if clean_str(
                content.get("publishedId")
            ) == requested_published_id:
                return hit, content

    if len(hits) == 1:
        hit = hits[0]
        content = hit.get("content") or {}

        requested_aff_id = clean_str(
            raw_rec.get("affId")
        )

        if (
            not requested_aff_id
            or clean_str(content.get("affId")) == requested_aff_id
        ):
            return hit, content

    return None, {}

def extract_inner_document_hits(hit):
    if not isinstance(hit, dict):
        return []
    inner = hit.get("innerHits") or {}
    doc_block = inner.get("document") or {}
    return doc_block.get("searchHits") or []

def extract_available_langs_from_content_ml(content_ml):
    if isinstance(content_ml, str):
        try:
            content_ml = json.loads(content_ml)
        except Exception:
            return []
    langs = []
    if isinstance(content_ml, list):
        for item in content_ml:
            if isinstance(item, dict):
                langs.extend([str(k).upper() for k in item.keys()])
    return sorted(set(langs))

def normalize_doc_formats(value):
    if value is None:
        return []
    if isinstance(value, list):
        vals = value
    elif isinstance(value, str):
        try:
            parsed = json.loads(value)
            vals = parsed if isinstance(parsed, list) else [value]
        except Exception:
            vals = re.split(r"[|,; ]+", value)
    else:
        vals = [value]
    return sorted(set([str(x).upper().strip() for x in vals if str(x).strip()]))

# Build input lookup once. This enriches raw JSON rows with fields from the original input manifest.
INPUT_BY_AFFID = {}
INPUT_BY_PROCEDURE_ID = {}
INPUT_BY_PUBLISHED_ID = {}

if not df_cases.empty:
    for _, _row in df_cases.iterrows():
        d = _row.to_dict()
        for col, target in [
            ("affId", INPUT_BY_AFFID),
            ("procedureId", INPUT_BY_PROCEDURE_ID),
            ("publishedId", INPUT_BY_PUBLISHED_ID),
            ("case_number_raw", INPUT_BY_PUBLISHED_ID),
        ]:
            key = clean_str(_row.get(col)) if col in df_cases.columns else ""
            if key and key not in target:
                target[key] = d

def lookup_input_row(raw_rec, content):
    """
    Resolve the upstream manifest row using procedureId first.
    """
    for key, mapping in [
        (
            clean_str(
                raw_rec.get("procedureId")
                or content.get("procedureId")
            ),
            INPUT_BY_PROCEDURE_ID,
        ),
        (
            clean_str(
                raw_rec.get("publishedId")
                or content.get("publishedId")
            ),
            INPUT_BY_PUBLISHED_ID,
        ),
        (
            clean_str(
                raw_rec.get("affId")
                or content.get("affId")
            ),
            INPUT_BY_AFFID,
        ),
    ]:
        if key and key in mapping:
            return mapping[key]

    return {}

def flatten_case_record(raw_rec):
    response = raw_rec.get("response") or {}
    hit, c = select_requested_procedure_hit(raw_rec)

    if not c:
        raise ValueError(
            "No response hit matched requested procedureId="
            f"{raw_rec.get('procedureId')!r}, "
            f"publishedId={raw_rec.get('publishedId')!r}, "
            f"affId={raw_rec.get('affId')!r}"
        )

    input_row = lookup_input_row(raw_rec, c)

    published = c.get("publishedId") or input_row.get("publishedId") or input_row.get("case_number_raw") or raw_rec.get("publishedId")
    parsed = parse_case_number(published, input_row.get("case_number_clean"), input_row.get("case_prefix"))

    calendars = c.get("calendars") or []
    delivery_dates = sorted({x.get("seanceDate") for x in calendars if isinstance(x, dict) and x.get("seanceDate")})
    delivery_types = sorted({x.get("seanceType") for x in calendars if isinstance(x, dict) and x.get("seanceType")})

    # Keep the existing notebook-level procedure_key for compatibility, but also expose
    # an explicit InfoCuria slash key in the requested format:
    #   infocuria_slash_procedure_key_base = YYYY/NNNN/PREFIX/
    #   infocuria_slash_procedure_key      = YYYY/NNNN/PREFIX/ or YYYY/NNNN/PREFIX/SUFFIX
    # This makes the matching key auditable even if upstream InfoCuria metadata already
    # contains another field called procedure_key.
    infocuria_slash_procedure_key = parsed.get("procedure_key") or ""
    infocuria_slash_procedure_key_base = parsed.get("procedure_key_base") or ""

    row = {
        "internal_key": normalize_internal_key(input_row.get("internal_key") or raw_rec.get("internal_key") or raw_rec.get("procedure_key") or input_row.get("procedure_key") or infocuria_slash_procedure_key),
        "procedure_key": normalize_internal_key(input_row.get("internal_key") or raw_rec.get("internal_key") or raw_rec.get("procedure_key") or input_row.get("procedure_key") or infocuria_slash_procedure_key),
        "procedure_key_base": internal_key_base(input_row.get("internal_key") or raw_rec.get("internal_key") or raw_rec.get("procedure_key") or input_row.get("procedure_key") or infocuria_slash_procedure_key) or infocuria_slash_procedure_key_base or input_row.get("procedure_key_base"),
        "infocuria_slash_procedure_key": infocuria_slash_procedure_key,
        "infocuria_slash_procedure_key_base": infocuria_slash_procedure_key_base,
        "case_number_raw": published,
        "case_number_clean": input_row.get("case_number_clean"),
        "case_prefix": parsed.get("case_prefix"),
        "case_number_int": parsed.get("case_number_int"),
        "case_number_int_full": parsed.get("case_number_int_full"),
        "case_year_raw": parsed.get("case_year_raw"),
        "case_year_full": parsed.get("case_year_full"),
        "case_suffix": parsed.get("case_suffix"),
        "affId": c.get("affId") or raw_rec.get("affId") or input_row.get("affId"),
        "publishedId": c.get("publishedId") or published,
        "publishedAffId": c.get("publishedAffId"),
        "procedureId": c.get("procedureId") or raw_rec.get("procedureId") or input_row.get("procedureId"),
        "jurisdiction": c.get("jurisdiction"),
        "jurisdictionCode": c.get("jurisdictionCode"),
        "nature": c.get("nature"),
        "natureCode": c.get("natureCode"),
        "procType": c.get("procType"),
        "procNumber": c.get("procNumber"),
        "suiteNumber": c.get("suiteNumber"),
        "case_name_en": ml_value(c.get("usualNameML"), "en"),

        "case_object_en": ml_value(c.get("affObjectML"), "en"),

        "introductionDate": c.get("introductionDate"),
        "procIntroDate": c.get("procIntroDate"),
        "introductionYear": c.get("introductionYear"),
        "closeDate": c.get("closeDate"),
        "procClosDate": c.get("procClosDate"),
        "closeYear": c.get("closeYear"),
        "affairState": c.get("affairState"),
        "affairStateCode": c.get("affairStateCode"),
        "procState": c.get("procState"),
        "procLang": c.get("procLang"),
        "joinExist": c.get("joinExist"),
        "pilot": c.get("pilot"),
        "idPilot": c.get("idPilot"),
        "pilot_case_publishedId": parse_affair_ref_list([c.get("idPilot")])[0] if c.get("idPilot") and parse_affair_ref_list([c.get("idPilot")]) else "",
        "joinAffairs": as_json_string(c.get("joinAffairs") or []),
        "joinAffairs_publishedIds": as_json_string(parse_affair_ref_list(c.get("joinAffairs") or [])),
        "pourvoiAffIds": as_json_string(c.get("pourvoiAffIds") or []),
        "pourvoi_publishedIds": as_json_string(parse_affair_ref_list(c.get("pourvoiAffIds") or [])),
        "originPourvoiIds": as_json_string(c.get("originPourvoiIds") or []),
        "origin_pourvoi_publishedIds": as_json_string(parse_affair_ref_list(c.get("originPourvoiIds") or [])),
        "reExamenAffIds": as_json_string(c.get("reExamenAffIds") or []),
        "originReExamanIds": as_json_string(c.get("originReExamanIds") or []),
        "transfertAffIds": as_json_string(c.get("transfertAffIds") or []),
        "originTransfertIds": as_json_string(c.get("originTransfertIds") or []),
        "matCode": as_json_string(c.get("matCode") or []),
        "subject_matter_en": label_text(c.get("matCodeML"), "en"),

        "reportingJudge": c.get("reportingJudge"),
        "formation": c.get("formation"),
        "advocate_general_code": c.get("avg"),
        "resultTypes": as_json_string(c.get("resultTypes") or []),
        "procedure_result_en": label_text(c.get("procedureResultTypeML"), "en"),

        "parties": c.get("parties"),
        "conclsDate": as_json_string(c.get("conclsDate") or []),
        "conclLang": c.get("conclLang"),
        "delivery_dates": as_json_string(delivery_dates),
        "delivery_types": as_json_string(delivery_types),
        "numDocs": c.get("numDocs"),
        "doctrineNotes_count": len(c.get("doctrineNotes") or []),
        "lastModifiedDate": c.get("lastModifiedDate"),
        "totalHits": response.get("totalHits"),
        "inner_document_totalHits": (((hit or {}).get("innerHits") or {}).get("document") or {}).get("totalHits"),
        "raw_case_content_json": json.dumps(c, ensure_ascii=False) if KEEP_RAW_CONTENT_JSON else "",
    }
    return row

def flatten_document_records(raw_rec):
    response = raw_rec.get("response") or {}
    hit, c = select_requested_procedure_hit(raw_rec)

    if not c:
        raise ValueError(
            "No document-bearing response hit matched requested procedureId="
            f"{raw_rec.get('procedureId')!r}, "
            f"publishedId={raw_rec.get('publishedId')!r}, "
            f"affId={raw_rec.get('affId')!r}"
        )

    case_row = flatten_case_record(raw_rec)
    docs = extract_inner_document_hits(hit)
    rows = []

    for pos, dh in enumerate(docs):
        dc = dh.get("content") or {}
        content_ml = dc.get("contentML") or []
        available_langs = extract_available_langs_from_content_ml(content_ml)
        doc_formats = normalize_doc_formats(dc.get("docFormats") or ([dc.get("docFormat")] if dc.get("docFormat") else []))
        code = normalize_doc_type(dc.get("docTypeCode") or dc.get("docType"))

        row = {
            "internal_key": case_row.get("internal_key"),
            "procedure_key": case_row.get("procedure_key"),
            "procedure_key_base": case_row.get("procedure_key_base"),
            "infocuria_slash_procedure_key": case_row.get("infocuria_slash_procedure_key"),
            "infocuria_slash_procedure_key_base": case_row.get("infocuria_slash_procedure_key_base"),
            "case_number_raw": case_row.get("case_number_raw"),
            "case_number_clean": case_row.get("case_number_clean"),
            "case_prefix": case_row.get("case_prefix"),
            "case_number_int": case_row.get("case_number_int"),
            "case_number_int_full": case_row.get("case_number_int_full"),
            "case_year_full": case_row.get("case_year_full"),
            "case_suffix": case_row.get("case_suffix"),
            "affId": case_row.get("affId"),
            "publishedId": case_row.get("publishedId"),
            "procedureId": case_row.get("procedureId"),
            "procLang": case_row.get("procLang"),
            "case_name_en": case_row.get("case_name_en"),
            "introductionDate": case_row.get("introductionDate"),
            "closeDate": case_row.get("closeDate"),
            "procIntroDate": case_row.get("procIntroDate"),
            "procClosDate": case_row.get("procClosDate"),
            "doc_position": pos,
            "inner_hit_id": dh.get("id"),
            "docId": dc.get("docId"),
            "logicDocId": dc.get("logicDocId"),
            "jpLogicDocId": dc.get("jpLogicDocId"),
            "idProcedure": dc.get("idProcedure"),
            "idPublished": dc.get("idPublished"),
            "docType": dc.get("docType"),
            "docTypeCode": dc.get("docTypeCode"),
            "normalized_docTypeCode": code,
            "relevance_flag": is_relevant_doc_type(code),
            "relevance_category": "relevant" if is_relevant_doc_type(code) else "irrelevant",
            "relevance_reason": relevance_reason(code),
            "lnsDec": dc.get("lnsDec"),
            "numDoc": dc.get("numDoc"),
            "docDate": dc.get("docDate"),
            "docFormat": dc.get("docFormat"),
            "docFormats": as_json_string(doc_formats),
            "docLang": str(dc.get("docLang") or "").upper(),
            "available_languages": as_json_string(available_langs),
            "docUrl": dc.get("docUrl"),
            "contentLength": dc.get("contentLength"),
            # Keep only English context text here. It is split into a separate compact context manifest
            # before writing final outputs, so the metadata manifest remains manageable.
            "context_text_en": ml_value(content_ml, "en", fallback=False),
            "raw_document_content_json": json.dumps(dc, ensure_ascii=False) if KEEP_RAW_CONTENT_JSON else "",
        }

        row["document_dedup_key"] = "|".join([
            clean_str(row.get("procedureId"))
            or clean_str(row.get("procedure_key")),
            clean_str(row.get("logicDocId"))
            or clean_str(row.get("docId"))
            or clean_str(row.get("inner_hit_id")),
            clean_str(row.get("normalized_docTypeCode")),
            clean_str(row.get("docDate")),
        ])
        rows.append(row)

    return rows


# ---------------------------------------------------------------------
# Stream raw JSONL into flattened case/document tables
# ---------------------------------------------------------------------

# Stream files are written directly under LOG_DIR and deleted after loading.
CASE_METADATA_STREAM_JSONL = LOG_DIR / "_case_metadata_stream.jsonl"
DOCUMENT_METADATA_STREAM_JSONL = LOG_DIR / "_document_metadata_stream.jsonl"

for p in [CASE_METADATA_STREAM_JSONL, DOCUMENT_METADATA_STREAM_JSONL, FLATTEN_FAILED_JSONL]:
    if p.exists():
        p.unlink()

def append_rows_jsonl(rows, path: Path):
    if not rows:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8", newline="\n") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

case_buffer = []
doc_buffer = []
failed_buffer = []
seen_cases = 0
seen_docs = 0

for rec in tqdm(iter_raw_jsonl(RAW_METADATA_JSONL), desc="Streaming flatten raw JSONL"):
    try:
        case_buffer.append(flatten_case_record(rec))
        doc_rows = flatten_document_records(rec)
        doc_buffer.extend(doc_rows)
        seen_cases += 1
        seen_docs += len(doc_rows)
    except Exception as e:
        failed_buffer.append({
            "affId": rec.get("affId"),
            "publishedId": rec.get("publishedId"),
            "procedureId": rec.get("procedureId"),
            "internal_key": rec.get("internal_key"),
            "procedure_key": rec.get("procedure_key"),
            "error": "flatten_failed: " + repr(e),
        })

    if len(case_buffer) >= STREAM_JSONL_CHUNK_SIZE:
        append_rows_jsonl(case_buffer, CASE_METADATA_STREAM_JSONL)
        case_buffer = []
    if len(doc_buffer) >= STREAM_JSONL_CHUNK_SIZE:
        append_rows_jsonl(doc_buffer, DOCUMENT_METADATA_STREAM_JSONL)
        doc_buffer = []
    if len(failed_buffer) >= STREAM_JSONL_CHUNK_SIZE:
        append_rows_jsonl(failed_buffer, FLATTEN_FAILED_JSONL)
        failed_buffer = []

append_rows_jsonl(case_buffer, CASE_METADATA_STREAM_JSONL)
append_rows_jsonl(doc_buffer, DOCUMENT_METADATA_STREAM_JSONL)
append_rows_jsonl(failed_buffer, FLATTEN_FAILED_JSONL)

print("Streamed case records:", f"{seen_cases:,}")
print("Streamed document records:", f"{seen_docs:,}")
print("Flatten failures:", FLATTEN_FAILED_JSONL if FLATTEN_FAILED_JSONL.exists() else "none")

df_case_flat = read_manifest(CASE_METADATA_STREAM_JSONL) if CASE_METADATA_STREAM_JSONL.exists() else pd.DataFrame()
df_doc_meta = read_manifest(DOCUMENT_METADATA_STREAM_JSONL) if DOCUMENT_METADATA_STREAM_JSONL.exists() else pd.DataFrame()

# Deduplicate internally.
#
# procedureId is the canonical case/procedure key. Rows with a populated
# procedureId are deduplicated only by procedureId. Fallback identifiers
# are used only for rows where procedureId is missing.
if not df_case_flat.empty:
    if "procedureId" in df_case_flat.columns:
        has_procedure_id = (
            df_case_flat["procedureId"]
            .fillna("")
            .astype(str)
            .str.strip()
            .ne("")
        )

        with_procedure_id = (
            df_case_flat.loc[has_procedure_id]
            .drop_duplicates(subset=["procedureId"], keep="last")
        )

        without_procedure_id = df_case_flat.loc[~has_procedure_id].copy()

        fallback_cols = [
            c
            for c in ["affId", "procedure_key", "internal_key"]
            if c in without_procedure_id.columns
        ]

        if fallback_cols:
            without_procedure_id = without_procedure_id.drop_duplicates(
                subset=fallback_cols,
                keep="last",
            )

        df_case_flat = pd.concat(
            [with_procedure_id, without_procedure_id],
            ignore_index=True,
        ).reset_index(drop=True)

    else:
        fallback_cols = [
            c
            for c in ["affId", "procedure_key", "internal_key"]
            if c in df_case_flat.columns
        ]

        if fallback_cols:
            df_case_flat = df_case_flat.drop_duplicates(
                subset=fallback_cols,
                keep="last",
            ).reset_index(drop=True)

if not df_doc_meta.empty and "document_dedup_key" in df_doc_meta.columns:
    df_doc_meta = df_doc_meta.drop_duplicates(
        subset=["document_dedup_key"],
        keep="last",
    ).reset_index(drop=True)

# Remove large context/raw fields and non-English fields from document metadata.
large_doc_cols = [
    "context_text_en", "content_text_en", "content_text_de", "content_text_fr", "content_text_best", "raw_document_content_json"
]
df_doc_meta = df_doc_meta.drop(columns=[c for c in large_doc_cols + DROP_NON_ENGLISH_METADATA_COLUMNS if c in df_doc_meta.columns], errors="ignore")
df_case_flat = df_case_flat.drop(columns=[c for c in DROP_NON_ENGLISH_METADATA_COLUMNS if c in df_case_flat.columns], errors="ignore")

# Remove temporary stream files. Final manifests are written later.
for p in [CASE_METADATA_STREAM_JSONL, DOCUMENT_METADATA_STREAM_JSONL]:
    try:
        if p.exists():
            p.unlink()
    except Exception as e:
        print(f"Could not delete temporary stream file {p}: {e}")

print("Case metadata rows after dedupe:", df_case_flat.shape)
print("Document metadata rows after dedupe, without context text:", df_doc_meta.shape)
display(df_case_flat.head(3))
display(df_doc_meta.head(3))

# ---------------------------------------------------------------------
# Document metadata manifest base
# ---------------------------------------------------------------------

# The final document metadata manifest is df_doc_meta after relevance columns are assigned.
# It intentionally excludes context text and raw document JSON.
print("Document metadata base rows:", len(df_doc_meta))
if not df_doc_meta.empty:
    display(df_doc_meta.head(10))

# ---------------------------------------------------------------------
# 2) Filtered relevant document metadata manifest
# ---------------------------------------------------------------------

if df_doc_meta.empty:
    df_relevant_docs = df_doc_meta.copy()
else:
    df_doc_meta["normalized_docTypeCode"] = df_doc_meta["normalized_docTypeCode"].map(normalize_doc_type)
    df_doc_meta["relevance_flag"] = df_doc_meta["normalized_docTypeCode"].map(is_relevant_doc_type)
    df_doc_meta["relevance_category"] = df_doc_meta["relevance_flag"].map({True: "relevant", False: "irrelevant"})
    df_doc_meta["relevance_reason"] = df_doc_meta["normalized_docTypeCode"].map(relevance_reason)
    df_doc_meta["relevant_doc_type_tokens"] = df_doc_meta["normalized_docTypeCode"].map(lambda x: "|".join(relevant_doc_type_tokens(x)))

    df_relevant_docs = df_doc_meta[df_doc_meta["relevance_flag"]].copy().reset_index(drop=True)

print("Relevant document rows:", len(df_relevant_docs))
if not df_relevant_docs.empty:
    display(df_relevant_docs[[
        c for c in ["procedure_key", "procedure_key_base", "infocuria_slash_procedure_key", "infocuria_slash_procedure_key_base", "publishedId", "logicDocId", "docType", "docTypeCode", "docDate", "available_languages", "docFormats", "relevance_reason"]
        if c in df_relevant_docs.columns
    ]].head(20))


# ---------------------------------------------------------------------
# 3) Case-level document availability aggregation and case metadata join
# ---------------------------------------------------------------------

def bool_series(s):
    return s.astype(str).str.lower().isin(["true", "1", "yes"])

def normalized_date_join(values, sep="|"):
    vals = []
    for v in values:
        s = clean_str(v)
        if not s or s.lower() in {"nan", "none", "<na>"}:
            continue
        try:
            dt = pd.to_datetime(s, errors="coerce", dayfirst=False)
            vals.append(dt.strftime("%Y-%m-%d") if pd.notna(dt) else s)
        except Exception:
            vals.append(s)
    return sep.join(sorted(set(vals)))

def best_join_key_cols(left, right):
    """
    Return the best shared identifier for joining two tables.

    procedureId is the canonical key. affId is only a fallback when
    procedureId is unavailable in either table.
    """
    for cols in [
        ["procedureId"],
        ["affId"],
        ["internal_key"],
        ["procedure_key"],
    ]:
        if all(c in left.columns and c in right.columns for c in cols):
            left_nonempty = left[cols[0]].fillna("").astype(str).str.strip().ne("").any()
            right_nonempty = right[cols[0]].fillna("").astype(str).str.strip().ne("").any()
            if left_nonempty and right_nonempty:
                return cols
    return []

if df_doc_meta.empty:
    doc_case_agg = pd.DataFrame(columns=[
        "procedureId", "affId", "internal_key", "procedure_key",
        "available_document_types", "relevant_document_types",
        "available_document_dates", "relevant_document_dates",
        "total_document_count", "relevant_document_count", "irrelevant_document_count",
        "case_relevance_flag",
    ])
else:
    df_doc_meta["relevance_flag"] = bool_series(df_doc_meta["relevance_flag"]) if "relevance_flag" in df_doc_meta.columns else False
    if (
        "procedureId" in df_doc_meta.columns
        and df_doc_meta["procedureId"]
            .fillna("")
            .astype(str)
            .str.strip()
            .ne("")
            .any()
    ):
        group_col = "procedureId"
    elif (
        "affId" in df_doc_meta.columns
        and df_doc_meta["affId"]
            .fillna("")
            .astype(str)
            .str.strip()
            .ne("")
            .any()
    ):
        group_col = "affId"
        print(
            "Warning: procedureId unavailable for document aggregation; "
            "falling back to affId."
        )
    else:
        group_col = next(
            (
                c
                for c in ["internal_key", "procedure_key"]
                if c in df_doc_meta.columns
            ),
            "procedure_key",
        )
    count_col = "document_dedup_key" if "document_dedup_key" in df_doc_meta.columns else "logicDocId"

    agg_kwargs = {
        "available_document_types": ("normalized_docTypeCode", unique_join),
        "total_document_count": (count_col, "count"),
        "relevant_document_count": ("relevance_flag", "sum"),
    }
    if "docDate" in df_doc_meta.columns:
        agg_kwargs["available_document_dates"] = ("docDate", normalized_date_join)

    doc_case_agg = df_doc_meta.groupby(group_col, dropna=False).agg(**agg_kwargs).reset_index()

    # Add representative identifiers for easier downstream joins/inspection.
    id_cols = [c for c in ["procedureId", "affId", "internal_key", "procedure_key", "procedure_key_base"] if c in df_doc_meta.columns and c != group_col]
    if id_cols:
        id_rep = df_doc_meta.groupby(group_col, dropna=False)[id_cols].agg(unique_join).reset_index()
        doc_case_agg = doc_case_agg.merge(id_rep, on=group_col, how="left")

    rel_doc = df_doc_meta[df_doc_meta["relevance_flag"]].copy()
    rel_types = rel_doc.groupby(group_col, dropna=False)["normalized_docTypeCode"].apply(unique_join).reset_index(name="relevant_document_types")
    doc_case_agg = doc_case_agg.merge(rel_types, on=group_col, how="left")

    if "docDate" in df_doc_meta.columns:
        rel_dates = rel_doc.groupby(group_col, dropna=False)["docDate"].apply(normalized_date_join).reset_index(name="relevant_document_dates")
        doc_case_agg = doc_case_agg.merge(rel_dates, on=group_col, how="left")

    for col in ["relevant_document_types", "available_document_dates", "relevant_document_dates"]:
        if col not in doc_case_agg.columns:
            doc_case_agg[col] = ""
        doc_case_agg[col] = doc_case_agg[col].fillna("")

    doc_case_agg["irrelevant_document_count"] = doc_case_agg["total_document_count"] - doc_case_agg["relevant_document_count"]
    doc_case_agg["case_relevance_flag"] = doc_case_agg["relevant_document_count"].gt(0)

# Start case manifest with full InfoCuria case skeleton, then left-join procedural metadata and document availability.
df_case_metadata_manifest = df_cases.copy()

# Join flattened procedural metadata by stable InfoCuria identifiers, not only by reconstructed keys.
metadata_join_keys = best_join_key_cols(df_case_metadata_manifest, df_case_flat)
if not df_case_flat.empty and metadata_join_keys:
    metadata_cols = [c for c in df_case_flat.columns if c not in df_case_metadata_manifest.columns or c in metadata_join_keys]
    df_case_metadata_manifest = df_case_metadata_manifest.merge(
        df_case_flat[metadata_cols].drop_duplicates(metadata_join_keys, keep="last"),
        on=metadata_join_keys,
        how="left",
        suffixes=("", "__metadata"),
    )
else:
    df_case_metadata_manifest["metadata_join_status"] = "no_raw_metadata_jsonl_records_or_no_join_key"

if "metadata_join_status" not in df_case_metadata_manifest.columns:
    joined_indicator_col = next((c for c in ["totalHits", "publishedAffId", "jurisdictionCode", "numDocs"] if c in df_case_metadata_manifest.columns), None)
    metadata_joined = df_case_metadata_manifest[joined_indicator_col].notna() if joined_indicator_col else pd.Series(False, index=df_case_metadata_manifest.index)
    df_case_metadata_manifest["metadata_join_status"] = metadata_joined.map({True: "metadata_joined", False: "no_metadata_join"})

# Join document aggregation by procedureId first. affId is only a fallback.
doc_join_keys = best_join_key_cols(df_case_metadata_manifest, doc_case_agg)
if not doc_case_agg.empty and doc_join_keys:
    doc_agg_cols = [c for c in doc_case_agg.columns if c not in df_case_metadata_manifest.columns or c in doc_join_keys]
    df_case_metadata_manifest = df_case_metadata_manifest.merge(doc_case_agg[doc_agg_cols], on=doc_join_keys, how="left")
else:
    print("Warning: no suitable key found for document aggregation join.")

for col in ["available_document_types", "relevant_document_types", "available_document_dates", "relevant_document_dates"]:
    if col not in df_case_metadata_manifest.columns:
        df_case_metadata_manifest[col] = ""
    df_case_metadata_manifest[col] = df_case_metadata_manifest[col].fillna("")

for col in ["total_document_count", "relevant_document_count", "irrelevant_document_count"]:
    if col not in df_case_metadata_manifest.columns:
        df_case_metadata_manifest[col] = 0
    df_case_metadata_manifest[col] = pd.to_numeric(df_case_metadata_manifest[col], errors="coerce").fillna(0).astype(int)

df_case_metadata_manifest["case_relevance_flag"] = df_case_metadata_manifest["relevant_document_count"].gt(0)
df_case_metadata_manifest = df_case_metadata_manifest.drop(columns=[c for c in DROP_NON_ENGLISH_METADATA_COLUMNS if c in df_case_metadata_manifest.columns], errors="ignore")

print("Case metadata manifest rows:", len(df_case_metadata_manifest))
print("Metadata join keys used:", metadata_join_keys)
print("Document aggregation join keys used:", doc_join_keys)
display(df_case_metadata_manifest[[
    c for c in ["internal_key", "procedure_key", "procedure_key_base", "publishedId", "affId", "procedureId", "available_document_types", "relevant_document_types", "available_document_dates", "relevant_document_dates", "total_document_count", "relevant_document_count", "irrelevant_document_count", "case_relevance_flag"]
    if c in df_case_metadata_manifest.columns
]].head(20))


## 5. Record joined proceedings

Use the structured joined-case fields and the additional procedural lookup to find associated case numbers. Keep each discovered procedure in the case table and record joined relationships in its fields; a joined set does not become one replacement row. The JSONL checkpoint lets this step resume.


In [ ]:
def normalize_joined_case_number(x):
    """Normalize a displayed InfoCuria case number, preserving suffixes such as P."""
    x = clean_str(x).upper()
    x = re.sub(r"<[^>]+>", " ", x)
    x = re.sub(r"&nbsp;|&#160;", " ", x, flags=re.I)
    x = re.sub(r"\s+", " ", x)
    x = re.sub(r"^([CTF])\s*-\s*", r"\1-", x)
    x = re.sub(r"\s*/\s*", "/", x)
    return x.strip()


JOINED_CASE_NUMBER_RE = re.compile(
    r"\b(?:C|T|F)-\s*\d+/\d+(?:\s*(?:P|PPU|R|DEP|P-DEP|REC|RX))?\b",
    flags=re.I,
)


def extract_case_after_hash(value):
    """
    Extract the displayed case number from an InfoCuria joined-affair reference.

    Expected shape:
        C/0007/56/00000000RF/01/P/01#C-7/56
    Returns:
        C-7/56
    """
    if value is None:
        return ""
    s = clean_str(value)
    if not s:
        return ""
    if "#" in s:
        return normalize_joined_case_number(s.split("#", 1)[1])
    # Conservative fallback for unexpected shapes.
    m = JOINED_CASE_NUMBER_RE.search(s)
    return normalize_joined_case_number(m.group(0)) if m else ""


def parse_join_affairs_value(value):
    """Parse joinAffairs whether it is already a list or a JSON/stringified list."""
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if isinstance(value, str):
        s = value.strip()
        if not s or s in {"[]", "nan", "None"}:
            return []
        try:
            parsed = json.loads(s)
            if isinstance(parsed, list):
                return parsed
            if parsed:
                return [parsed]
            return []
        except Exception:
            # Allow pipe-separated or comma-separated diagnostics as fallback.
            if "|" in s:
                return [x.strip() for x in s.split("|") if x.strip()]
            if "," in s:
                return [x.strip() for x in s.split(",") if x.strip()]
            return [s]
    return [value]


def extract_joined_cases_from_content(content: dict, query_case: str):
    """
    Production joined-case extraction rule.

    Use only:
      - content.idPilot
      - content.joinAffairs

    Do not inspect doctrine notes, full document text, highlights, or arbitrary JSON.
    """
    query_norm = normalize_joined_case_number(query_case)
    raw_sources = []

    id_pilot = content.get("idPilot")
    if id_pilot:
        raw_sources.append(("idPilot", id_pilot))

    for v in parse_join_affairs_value(content.get("joinAffairs") or []):
        if v:
            raw_sources.append(("joinAffairs", v))

    joined_cases = []
    for source, value in raw_sources:
        c = extract_case_after_hash(value)
        if c and c != query_norm:
            joined_cases.append(c)

    # Deduplicate while preserving API order.
    seen = set()
    ordered = []
    for c in joined_cases:
        if c not in seen:
            seen.add(c)
            ordered.append(c)

    return ordered, raw_sources


def procedures_payload_for_joined_cases(aff_id: str, published_id: str, language: str = LANGUAGE):
    return {
        "affId": clean_str(aff_id),
        "searchTerm": f'"{clean_str(published_id)}"',
        "tabName": "affair",
        "language": language,
    }


def choose_procedure_content_for_affid(response_json: dict, aff_id: str, published_id: str = ""):
    """Choose the exact searchHits[*].content for the requested affId/publishedId."""
    hits = response_json.get("searchHits") or []
    if not hits:
        return {}, "no_hits"

    aff_id = clean_str(aff_id)
    published_norm = normalize_joined_case_number(published_id)

    # Prefer exact affId + exact publishedId/publishedAffId.
    for hit in hits:
        c = hit.get("content") or {}
        if clean_str(c.get("affId")) != aff_id:
            continue
        if published_norm and normalize_joined_case_number(c.get("publishedId")) == published_norm:
            return c, "exact_affid_publishedId"
        if published_norm and normalize_joined_case_number(c.get("publishedAffId")) == published_norm:
            return c, "exact_affid_publishedAffId"

    # Then exact affId only.
    for hit in hits:
        c = hit.get("content") or {}
        if clean_str(c.get("affId")) == aff_id:
            return c, "exact_affid"

    # Last fallback: first hit.
    return (hits[0].get("content") or {}), "first_hit_fallback"


def local_joined_cases_from_row(row: pd.Series, published_id: str):
    """Extract joined cases from already-flattened local row columns if available."""
    content = {
        "idPilot": row.get("idPilot", ""),
        "joinAffairs": row.get("joinAffairs", []),
    }
    return extract_joined_cases_from_content(content, published_id)


def display_case_for_join_collection(row: pd.Series, candidate_cols=None):
    """Pick the best displayed case identifier for joined-case grouping."""
    if candidate_cols is None:
        candidate_cols = [
            "publishedId", "case_number_raw", "idPublished", "publishedAffId",
            "case_number_clean", "procNumber", "case_number",
        ]
    for c in candidate_cols:
        if c in row.index:
            v = normalize_joined_case_number(row.get(c))
            if JOINED_CASE_NUMBER_RE.search(v):
                return v
            # Some columns may already be exactly normalized but fail due to suffix quirks.
            if re.match(r"^(?:C|T|F)-\d+/\d+", v):
                return v
    return ""


def split_pipe_cases(value):
    """Split a pipe/list-like case cell into normalized case numbers."""
    s = clean_str(value)
    if not s:
        return []
    values = []
    # JSON/list-style cells are tolerated.
    if s.startswith("["):
        try:
            obj = json.loads(s)
            if isinstance(obj, list):
                values = obj
            else:
                values = [obj]
        except Exception:
            values = re.split(r"[|,;]", s)
    else:
        values = re.split(r"[|,;]", s)
    out = []
    for v in values:
        c = normalize_joined_case_number(v)
        if c and re.match(r"^(?:C|T|F)-\d+/\d+", c):
            out.append(c)
    # dedupe preserving order
    seen = set()
    ordered = []
    for c in out:
        if c not in seen:
            seen.add(c)
            ordered.append(c)
    return ordered


def add_joined_case_collection(df_cases_manifest: pd.DataFrame) -> pd.DataFrame:
    """
    Add propagated joined-case groups after joined_case_search.

    Example:
      C-1/00 row says C-2/00|C-3/00
      then rows C-1/00, C-2/00 and C-3/00 all receive:
      C-1/00|C-2/00|C-3/00

    This does not replace the direct `joined_case_search`; it adds a new
    aggregate column. Existing values in `joined_case_collection` are
    merged with newly inferred values rather than discarded.
    """
    out = df_cases_manifest.copy()
    collection_col = "joined_case_collection"
    if collection_col not in out.columns:
        out[collection_col] = ""

    row_cases = out.apply(display_case_for_join_collection, axis=1)
    out["_join_collection_case_number"] = row_cases

    parent = {}
    order_seen = []

    def add_node(x):
        x = normalize_joined_case_number(x)
        if not x:
            return ""
        if x not in parent:
            parent[x] = x
            order_seen.append(x)
        return x

    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        a = add_node(a)
        b = add_node(b)
        if not a or not b:
            return
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    # Build links: row's own displayed case + all directly extracted joined cases.
    for idx, row in out.iterrows():
        own_case = row_cases.loc[idx]
        direct_joined = split_pipe_cases(row.get("joined_case_search", ""))
        existing_collection = split_pipe_cases(row.get(collection_col, ""))
        group = [own_case] + direct_joined + existing_collection
        group = [g for g in group if g]
        if not group:
            continue
        add_node(group[0])
        for g in group[1:]:
            union(group[0], g)

    # Collect connected components preserving discovery order.
    components = {}
    for case in order_seen:
        root = find(case)
        components.setdefault(root, [])
        if case not in components[root]:
            components[root].append(case)

    case_to_collection = {}
    for cases in components.values():
        joined = "|".join(cases)
        for c in cases:
            case_to_collection[c] = joined

    # Assign propagated collection to every row with a known case.
    for idx, own_case in row_cases.items():
        existing = split_pipe_cases(out.at[idx, collection_col])
        propagated = split_pipe_cases(case_to_collection.get(own_case, ""))
        merged = []
        for c in existing + propagated:
            if c and c not in merged:
                merged.append(c)
        out.at[idx, collection_col] = "|".join(merged)

    populated = out[collection_col].fillna("").astype(str).str.strip().ne("").sum()
    print(f"Rows with joined_case_collection populated: {populated:,}")
    out.drop(columns=["_join_collection_case_number"], inplace=True, errors="ignore")
    return out



def joined_case_request_key(aff_id: str, published_id: str) -> str:
    """Stable key for the joined-case API cache."""
    return f"{clean_str(aff_id)}||{normalize_joined_case_number(published_id)}"


def joined_case_cache_record_key(record: dict) -> str:
    return clean_str(record.get("joined_case_request_key")) or joined_case_request_key(
        record.get("affId", ""),
        record.get("publishedId", ""),
    )


def load_joined_case_cache_latest(path: Path) -> dict:
    """Load latest JSONL record per affId/publishedId key, keeping the last attempt."""
    latest = {}
    if not path.exists():
        return latest
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except Exception:
                # Do not spam warnings during long runs; just ignore malformed cache lines.
                continue
            key = joined_case_cache_record_key(rec)
            if key:
                latest[key] = rec
    return latest


def append_joined_case_jsonl(path: Path, record: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8", newline="\n") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def compact_joined_case_jsonl_keep_last(path: Path):
    """Compact joined-case JSONL to one latest record per request key."""
    if not path.exists():
        return
    latest = load_joined_case_cache_latest(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8", newline="\n") as f:
        for rec in latest.values():
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    tmp.replace(path)
    print(f"Compacted joined-case JSONL to {len(latest):,} latest records -> {path}")


def joined_case_cache_record_is_reusable(record: dict) -> bool:
    """
    True means update_failed mode can reuse this cache record without calling the API again.

    We reuse HTTP 200/success records, including rows where the API worked but found no joined cases.
    We retry explicit failure statuses and retryable HTTP statuses such as 403/429/5xx.
    """
    if not record:
        return False
    status = clean_str(record.get("status")).lower()
    http_status = record.get("http_status")
    try:
        http_status_int = int(http_status) if clean_str(http_status) else None
    except Exception:
        http_status_int = None

    if status == "success" and http_status_int == 200:
        return True
    if http_status_int in JOINED_CASE_RETRY_HTTP_STATUSES:
        return False
    if status in {"http_error", "exception", "parse_error", "local_fallback_error", "missing_identifiers"}:
        return False
    return False


def response_summary_for_joined_case_cache(response_json: dict) -> dict:
    """Light diagnostics only; avoids storing the entire API payload for every row."""
    hits = response_json.get("searchHits") or [] if isinstance(response_json, dict) else []
    summary = {
        "search_hits_count": len(hits),
        "hit_affIds": [],
        "hit_publishedIds": [],
    }
    if JOINED_CASE_RECORD_RESPONSE_SUMMARY:
        for hit in hits[:10]:
            c = hit.get("content") or {}
            summary["hit_affIds"].append(clean_str(c.get("affId")))
            summary["hit_publishedIds"].append(first_nonempty(c.get("publishedId"), c.get("publishedAffId")))
    return summary


def build_joined_case_cache_record_from_local(row, idx, aff_id, published_id, reason="local_only"):
    """Create a non-API record from local flattened fields, used for disabled/csv/cache fallback cases."""
    local_cases, local_raw_sources = local_joined_cases_from_row(row, published_id)
    return {
        "joined_case_request_key": joined_case_request_key(aff_id, published_id),
        "row_index": int(idx) if isinstance(idx, (int, np.integer)) else str(idx),
        "affId": aff_id,
        "publishedId": published_id,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "status": reason,
        "http_status": None,
        "content_choice": "local_flattened_fields",
        "joined_case_search": "|".join(local_cases),
        "joined_case_search_source": "local_flattened_fields" if local_cases else "no_local_joined_cases",
        "joined_case_search_raw_sources": "|".join(f"{src}={val}" for src, val in local_raw_sources),
        "api_joined_cases": [],
        "local_joined_cases": local_cases,
        "error_type": "",
        "error": "",
    }


def apply_joined_case_cache_record_to_output(out: pd.DataFrame, idx, row, record: dict, published_id: str):
    """Apply one cache/API/local record to the case manifest row."""
    if not record:
        local_cases, local_raw_sources = local_joined_cases_from_row(row, published_id)
        out.at[idx, "joined_case_search"] = "|".join(local_cases)
        out.at[idx, "joined_case_search_source"] = "local_flattened_fields_no_cache" if local_cases else "no_cache_no_local_joined_cases"
        out.at[idx, "joined_case_search_raw_sources"] = "|".join(f"{src}={val}" for src, val in local_raw_sources)
        out.at[idx, "joined_case_search_status"] = "no_cache"
        return

    out.at[idx, "joined_case_search"] = clean_str(record.get("joined_case_search"))
    out.at[idx, "joined_case_search_source"] = clean_str(record.get("joined_case_search_source"))
    out.at[idx, "joined_case_search_raw_sources"] = clean_str(record.get("joined_case_search_raw_sources"))

    status = clean_str(record.get("status"))
    http_status = clean_str(record.get("http_status"))
    if http_status:
        out.at[idx, "joined_case_search_status"] = f"{status}:http_{http_status}"
    else:
        out.at[idx, "joined_case_search_status"] = status


def request_joined_case_record(session, row, idx, aff_id: str, published_id: str, request_counter: int) -> dict:
    """
    Call InfoCuria once and return a JSONL-safe result record.

    No warning is logged for 403/429/etc. These are normal, restartable outcomes.
    """
    local_cases, local_raw_sources = local_joined_cases_from_row(row, published_id)
    payload = procedures_payload_for_joined_cases(aff_id, published_id)
    base_record = {
        "joined_case_request_key": joined_case_request_key(aff_id, published_id),
        "row_index": int(idx) if isinstance(idx, (int, np.integer)) else str(idx),
        "request_counter": int(request_counter),
        "affId": aff_id,
        "publishedId": published_id,
        "payload": payload,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "local_joined_cases": local_cases,
        "joined_case_search_raw_sources": "|".join(f"{src}={val}" for src, val in local_raw_sources),
        "api_joined_cases": [],
        "content_choice": "",
        "response_summary": {},
    }

    try:
        resp = session.post(
            JOINED_CASE_PROCEDURES_URL,
            json=payload,
            timeout=METADATA_REQUEST_TIMEOUT_SECONDS,
        )
        base_record["http_status"] = int(resp.status_code)

        if resp.status_code >= 400:
            final_cases = local_cases
            base_record.update({
                "status": "http_error",
                "joined_case_search": "|".join(final_cases),
                "joined_case_search_source": (
                    "local_flattened_fields_after_api_error" if final_cases else "api_error_no_local_joined_cases"
                ),
                "error_type": "HTTPError",
                "error": f"HTTP {resp.status_code}: {resp.text[:300]}",
            })
            return base_record

        try:
            response_json = resp.json()
        except Exception as e:
            final_cases = local_cases
            base_record.update({
                "status": "parse_error",
                "joined_case_search": "|".join(final_cases),
                "joined_case_search_source": (
                    "local_flattened_fields_after_parse_error" if final_cases else "parse_error_no_local_joined_cases"
                ),
                "error_type": type(e).__name__,
                "error": repr(e),
            })
            return base_record

        content, content_choice = choose_procedure_content_for_affid(response_json, aff_id, published_id)
        api_cases, api_raw_sources = extract_joined_cases_from_content(content, published_id)
        final_cases = api_cases or local_cases
        final_raw_sources = api_raw_sources or local_raw_sources
        source = f"procedures_endpoint:{content_choice}" if api_cases else "local_flattened_fields"

        base_record.update({
            "status": "success",
            "content_choice": content_choice,
            "response_summary": response_summary_for_joined_case_cache(response_json),
            "api_joined_cases": api_cases,
            "joined_case_search": "|".join(final_cases),
            "joined_case_search_source": source if final_cases else f"no_joined_cases_found:{content_choice}",
            "joined_case_search_raw_sources": "|".join(f"{src}={val}" for src, val in final_raw_sources),
            "error_type": "",
            "error": "",
        })
        return base_record

    except Exception as e:
        final_cases = local_cases
        base_record.update({
            "status": "exception",
            "http_status": None,
            "joined_case_search": "|".join(final_cases),
            "joined_case_search_source": (
                "local_flattened_fields_after_exception" if final_cases else "exception_no_local_joined_cases"
            ),
            "error_type": type(e).__name__,
            "error": repr(e)[:500],
        })
        return base_record


def add_joined_case_search(df_cases_manifest: pd.DataFrame) -> pd.DataFrame:
    out = df_cases_manifest.copy()

    for col in [
        "joined_case_search",
        "joined_case_search_status",
        "joined_case_search_source",
        "joined_case_search_raw_sources",
        "joined_case_collection",
    ]:
        if col not in out.columns:
            out[col] = ""

    published_col = next(
        (c for c in ["publishedId", "case_number_raw", "idPublished", "case_number_clean"] if c in out.columns),
        None,
    )
    if published_col is None:
        print("Joined-case API lookup skipped: no published case identifier column found.")
        return add_joined_case_collection(out)

    aff_col = next((c for c in ["affId", "idAffaire"] if c in out.columns), None)
    if aff_col is None:
        print("Joined-case API lookup skipped: no affId column found.")
        return add_joined_case_collection(out)
    # This is intentional because asymmetric joined-case metadata can be carried back
    # to all members of a connected joined-case component.
    todo = out.loc[
        out[published_col].fillna("").astype(str).str.strip().ne("")
        & out[aff_col].fillna("").astype(str).str.strip().ne("")
    ].copy()

    if JOINED_CASE_SEARCH_LIMIT is not None:
        todo = todo.head(int(JOINED_CASE_SEARCH_LIMIT)).copy()

    print(f"Joined-case repair candidate rows, all cases with identifiers: {len(todo):,}")
    print("JOINED_CASE_JSONL_MODE:", JOINED_CASE_JSONL_MODE)
    print("Joined-case JSONL:", JOINED_CASE_JSONL)

    if todo.empty:
        return add_joined_case_collection(out)

    if not RUN_JOINED_CASE_SEARCH:
        print("Joined-case API lookup disabled; building collection from existing/manual/local fields only.")
        for idx, row in todo.iterrows():
            aff_id = clean_str(row.get(aff_col))
            published_id = clean_str(row.get(published_col))
            record = build_joined_case_cache_record_from_local(row, idx, aff_id, published_id, reason="local_only_api_disabled")
            apply_joined_case_cache_record_to_output(out, idx, row, record, published_id)
        return add_joined_case_collection(out)

    if JOINED_CASE_JSONL_MODE == "rebuild_jsonl" and JOINED_CASE_JSONL.exists():
        JOINED_CASE_JSONL.unlink()
        print("Deleted existing joined-case JSONL for rebuild:", JOINED_CASE_JSONL)

    cache_latest = load_joined_case_cache_latest(JOINED_CASE_JSONL)
    print(f"Loaded joined-case cache records: {len(cache_latest):,}")

    session = requests.Session()
    session.headers.update({
        "accept": "application/json",
        "accept-language": HEADERS.get("Accept-Language", "en-US,en;q=0.9"),
        "content-type": "application/json; charset=UTF-8",
        "origin": "https://infocuria.curia.europa.eu",
        "referer": "https://infocuria.curia.europa.eu/",
        "user-agent": HEADERS.get("User-Agent", "Mozilla/5.0"),
    })

    stats = Counter()
    api_request_count = 0

    if JOINED_CASE_JSONL_MODE == "csv_from_jsonl":
        for idx, row in tqdm(todo.iterrows(), total=len(todo), desc="Joined-case repair from JSONL only"):
            aff_id = clean_str(row.get(aff_col))
            published_id = clean_str(row.get(published_col))
            key = joined_case_request_key(aff_id, published_id)
            record = cache_latest.get(key)
            if record:
                stats[f"cache_{clean_str(record.get('status')) or 'unknown'}"] += 1
            else:
                stats["missing_cache_local_fallback"] += 1
            apply_joined_case_cache_record_to_output(out, idx, row, record, published_id)

        print("Joined-case JSONL-only rebuild summary:")
        print(pd.Series(dict(stats)).sort_index())
        found = out["joined_case_search"].fillna("").astype(str).str.strip().ne("").sum()
        print(f"Cases with joined_case_search populated: {found:,}")
        return add_joined_case_collection(out)

    rows_to_request = []
    rows_to_reuse = []
    for idx, row in todo.iterrows():
        aff_id = clean_str(row.get(aff_col))
        published_id = clean_str(row.get(published_col))
        key = joined_case_request_key(aff_id, published_id)
        cached = cache_latest.get(key)
        if JOINED_CASE_JSONL_MODE == "update_failed" and joined_case_cache_record_is_reusable(cached):
            rows_to_reuse.append((idx, row, cached))
        else:
            rows_to_request.append((idx, row))

    print(f"Joined-case rows reused from cache: {len(rows_to_reuse):,}")
    print(f"Joined-case rows to request now: {len(rows_to_request):,}")

    for idx, row, cached in tqdm(rows_to_reuse, total=len(rows_to_reuse), desc="Joined-case cache reuse"):
        published_id = clean_str(row.get(published_col))
        apply_joined_case_cache_record_to_output(out, idx, row, cached, published_id)
        stats[f"reused_{clean_str(cached.get('status')) or 'unknown'}"] += 1

    pbar = tqdm(rows_to_request, total=len(rows_to_request), desc="Joined-case API/cache update")
    for idx, row in pbar:
        aff_id = clean_str(row.get(aff_col))
        published_id = clean_str(row.get(published_col))

        api_request_count += 1
        record = request_joined_case_record(session, row, idx, aff_id, published_id, api_request_count)
        append_joined_case_jsonl(JOINED_CASE_JSONL, record)
        cache_latest[joined_case_cache_record_key(record)] = record
        apply_joined_case_cache_record_to_output(out, idx, row, record, published_id)

        status = clean_str(record.get("status")) or "unknown"
        http_status = clean_str(record.get("http_status"))
        stats[status] += 1
        if http_status:
            stats[f"http_{http_status}"] += 1
        pbar.set_postfix({
            "ok": stats.get("success", 0),
            "403": stats.get("http_403", 0),
            "429": stats.get("http_429", 0),
            "err": stats.get("exception", 0) + stats.get("parse_error", 0),
        })

        if JOINED_CASE_SLEEP_SECONDS:
            time.sleep(float(JOINED_CASE_SLEEP_SECONDS))

        if (
            JOINED_CASE_BATCH_PAUSE_EVERY
            and api_request_count % int(JOINED_CASE_BATCH_PAUSE_EVERY) == 0
            and api_request_count < len(rows_to_request)
        ):
            pause = float(JOINED_CASE_BATCH_PAUSE_SECONDS)
            print(f"\nJoined-case cooldown after {api_request_count:,} API requests: sleeping {pause:.0f}s")
            time.sleep(pause)

    if JOINED_CASE_COMPACT_JSONL_AFTER_RUN:
        compact_joined_case_jsonl_keep_last(JOINED_CASE_JSONL)

    print("Joined-case repair summary for this run:")
    if stats:
        print(pd.Series(dict(stats)).sort_index())
    else:
        print("No joined-case API requests were needed.")

    found = out["joined_case_search"].fillna("").astype(str).str.strip().ne("").sum()
    print(f"Cases with joined_case_search populated: {found:,}")
    out = add_joined_case_collection(out)
    return out


# Execute lookup before final manifests are saved.
df_case_metadata_manifest = add_joined_case_search(df_case_metadata_manifest)


## 6. Download selected documents

Try the preferred available languages and formats, reusing files already present. Record each attempt in `logs/court_infocuria/document_download_state.jsonl` so a stopped run can continue. The final download workbook connects source document identifiers to local raw-file paths for notebook 03.


In [ ]:
# ---------------------------------------------------------------------
# 5) Download URL and target-path builders
# ---------------------------------------------------------------------
# when downloading, but the final manifest stores only attempted URLs, the selected
# first URL, and the final working URL/status/path.

def parse_json_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            obj = json.loads(x)
            return obj if isinstance(obj, list) else []
        except Exception:
            return []
    return []


def ordered_unique(values):
    out = []
    seen = set()
    for v in values:
        s = clean_str(v)
        if s and s not in seen:
            seen.add(s)
            out.append(s)
    return out


def available_languages_for_row(row):
    langs = [str(x).upper() for x in parse_json_list(row.get("available_languages")) if x]
    proc_lang = clean_str(row.get("procLang")).upper()
    ordered = []
    for lang in PREFERRED_LANGUAGES:
        if lang.upper() in langs:
            ordered.append(lang.upper())
    if proc_lang and proc_lang in langs:
        ordered.append(proc_lang)
    ordered.extend(langs)
    return ordered_unique(ordered)


def available_formats_for_row(row):
    fmts = [str(x).upper() for x in parse_json_list(row.get("docFormats")) if x]
    if not fmts and clean_str(row.get("docFormat")):
        fmts = normalize_doc_formats(row.get("docFormat"))
    ordered = []
    for fmt in PREFERRED_FORMATS:
        if fmt.upper() in fmts:
            ordered.append(fmt.upper())
    ordered.extend(fmts)
    return ordered_unique(ordered)


def language_format_pairs_for_row(row):
    langs = available_languages_for_row(row) or ["EN"]
    fmts = available_formats_for_row(row) or ["HTML", "PDF"]
    pairs = []
    for lang in langs:
        for fmt in fmts:
            pairs.append((lang, fmt))
    return pairs


def year_from_date_value(value):
    s = clean_str(value)
    if not s:
        return ""
    m = re.search(r"(19|20)\d{2}", s)
    if m:
        return m.group(0)
    for dayfirst in [False, True]:
        try:
            dt = pd.to_datetime(s, errors="coerce", dayfirst=dayfirst)
            if pd.notna(dt):
                return f"{int(dt.year):04d}"
        except Exception:
            pass
    return ""


def parse_routing_parts(routing):
    parts = str(routing or "").strip().split("/")
    if len(parts) < 3:
        return None
    court, number, two_digit_year, *rest = parts
    if not court or not number or not two_digit_year:
        return None
    return court, number, two_digit_year, rest


def full_year_from_two_digit(two_digit_year):
    try:
        y = int(str(two_digit_year))
        return "19" + f"{y:02d}" if y >= 50 else "20" + f"{y:02d}"
    except Exception:
        return ""


def routing_case_year(routing):
    parsed = parse_routing_parts(routing)
    return full_year_from_two_digit(parsed[2]) if parsed else ""


def candidate_years_for_row(row):
    routing = first_nonempty(row.get("idProcedure"), row.get("procedureId"))
    base_years = []
    for y in [
        routing_case_year(routing),
        year_from_date_value(row.get("docDate")),
        year_from_date_value(row.get("closeDate")),
        year_from_date_value(row.get("procClosDate")),
        year_from_date_value(row.get("introductionDate")),
        year_from_date_value(row.get("procIntroDate")),
        clean_str(row.get("case_year_full")),
    ]:
        if re.fullmatch(r"(19|20)\d{2}", clean_str(y)):
            base_years.append(int(y))
    if not base_years:
        return []
    lo, hi = min(base_years), max(base_years)
    years = list(range(lo - 2, hi + 3))
    ordered = [int(y) for y in base_years] + years
    return [str(y) for y in ordered_unique([str(y) for y in ordered if 1950 <= int(y) <= 2035])]


def routing_to_blob_path(routing, override_year=None):
    parsed = parse_routing_parts(routing)
    if not parsed:
        return None
    court, number, two_digit_year, rest = parsed
    full_year = clean_str(override_year) or full_year_from_two_digit(two_digit_year)
    if not full_year:
        return None
    filename_part = f"{court}_{number}_{two_digit_year}_" + "_".join(rest)
    return f"{court}/{full_year}/{filename_part}"


def build_download_url(row, lang, fmt, override_year=None):
    logic_doc_id = clean_str(row.get("logicDocId"))
    routing = first_nonempty(row.get("idProcedure"), row.get("procedureId"))
    if not logic_doc_id or not routing or not lang or not fmt:
        return ""
    blob_path = routing_to_blob_path(routing, override_year=override_year)
    if not blob_path:
        return ""
    logic_num = logic_doc_id.replace("id_", "")
    fmt = str(fmt).upper()
    lang = str(lang).upper()
    if fmt == "HTML":
        return f"https://infocuriaws.curia.europa.eu/blob/download-file-html/{blob_path}/{logic_num}-{lang}-1.html"
    if fmt == "PDF":
        return f"https://infocuriaws.curia.europa.eu/blob/download-file/{blob_path}/{logic_num}-{lang}-1.pdf"
    return ""


def safe_filename(value, fallback="document"):
    value = "" if value is None else str(value)
    value = re.sub(r"[^A-Za-z0-9._-]+", "_", value).strip("_")
    return value or fallback


def filename_from_url(url, fallback):
    try:
        parsed = urlparse(str(url))
        name = Path(parsed.path).name
        return safe_filename(name, fallback=fallback)
    except Exception:
        return safe_filename(fallback)


def primary_target_dir_for_format(fmt):
    fmt = str(fmt or "").upper()
    if fmt == "HTML":
        return COURT_INFOCURIA_HTML_DIR
    if fmt == "PDF":
        return COURT_INFOCURIA_PDFS_DIR
    return COURT_INFOCURIA_RAW_DIR / "other"


def candidate_existing_dirs_for_format(fmt):
    fmt = str(fmt or "").upper()
    if fmt == "HTML":
        return COURT_INFOCURIA_HTML_EXISTING_DIRS
    if fmt == "PDF":
        return COURT_INFOCURIA_PDF_EXISTING_DIRS
    return [COURT_INFOCURIA_RAW_DIR / "other"]


def find_existing_download_path(fmt, filename):
    for folder in candidate_existing_dirs_for_format(fmt):
        candidate = Path(folder) / filename
        if candidate.exists():
            return candidate.resolve()
    return None


def build_candidate_download_records(row):
    records = []
    years = candidate_years_for_row(row) or [None]
    for lang, fmt in language_format_pairs_for_row(row):
        for year in years:
            url = build_download_url(row, lang, fmt, override_year=year)
            if not url:
                continue
            fallback_base = safe_filename("_".join([
                clean_str(row.get("procedure_key")),
                first_nonempty(row.get("logicDocId"), row.get("docId")),
                clean_str(row.get("normalized_docTypeCode")),
                clean_str(lang),
                clean_str(year),
            ]), fallback="infocuria_document")
            ext = "html" if fmt == "HTML" else "pdf" if fmt == "PDF" else "bin"
            filename = filename_from_url(url, fallback=f"{fallback_base}.{ext}")
            target_dir = primary_target_dir_for_format(fmt)
            target_dir.mkdir(parents=True, exist_ok=True)
            intended_local_path = target_dir / filename
            existing_path = find_existing_download_path(fmt, filename)
            records.append({
                "url": url,
                "language": lang,
                "format": fmt,
                "year": year,
                "filename": filename,
                "local_path": str(existing_path if existing_path is not None else intended_local_path),
                "intended_local_path": str(intended_local_path),
                "existing_local_path": str(existing_path) if existing_path is not None else "",
                "file_exists_before_download": bool(existing_path is not None or intended_local_path.exists()),
            })
    out = []
    seen = set()
    for r in records:
        if r["url"] not in seen:
            seen.add(r["url"])
            out.append(r)
    return out


def build_download_manifest(relevant_docs):
    """Build one download target row per relevant document.

    Every row in df_relevant_docs, including ORD_COMM and
    combined codes such as ORD_COMM|REQ_COMM, receives a download-manifest row.
    Candidate URL lists are generated only in memory and are not exported.
    """
    rows = []
    for _, row in relevant_docs.iterrows():
        candidate_records = build_candidate_download_records(row)
        first = candidate_records[0] if candidate_records else {}
        already_exists = bool(first.get("file_exists_before_download", False))
        rows.append({
            "internal_key": row.get("internal_key"),
            "procedure_key": row.get("procedure_key"),
            "procedure_key_base": row.get("procedure_key_base"),
            "case_number_raw": row.get("case_number_raw"),
            "case_number_clean": row.get("case_number_clean"),
            "affId": row.get("affId"),
            "procedureId": row.get("procedureId"),
            "publishedId": row.get("publishedId"),
            "docId": row.get("docId"),
            "logicDocId": row.get("logicDocId"),
            "jpLogicDocId": row.get("jpLogicDocId"),
            "idProcedure": row.get("idProcedure"),
            "idPublished": row.get("idPublished"),
            "docType": row.get("docType"),
            "docTypeCode": row.get("docTypeCode"),
            "normalized_docTypeCode": row.get("normalized_docTypeCode"),
            "docDate": row.get("docDate"),
            "available_languages": row.get("available_languages"),
            "docFormats": row.get("docFormats"),
            "candidate_years": json.dumps(candidate_years_for_row(row), ensure_ascii=False),
            "download_url": first.get("url", ""),
            "selected_language": first.get("language", ""),
            "selected_format": first.get("format", ""),
            "selected_year": first.get("year", ""),
            "target_filename": first.get("filename", ""),
            "intended_local_path": first.get("intended_local_path", ""),
            "existing_local_path": first.get("existing_local_path", ""),
            "final_local_path": first.get("local_path", "") if already_exists else pd.NA,
            "file_present_on_disk": already_exists,
            "file_exists_before_download": already_exists,
            "download_success": already_exists,
            "download_status": "already_exists" if already_exists else "pending",
            "attempted_urls": json.dumps([], ensure_ascii=False),
            "attempted_http_statuses": json.dumps([], ensure_ascii=False),
            "attempt_count": 0,
            "working_url": first.get("url", "") if already_exists else pd.NA,
            "working_language": first.get("language", "") if already_exists else pd.NA,
            "working_format": first.get("format", "") if already_exists else pd.NA,
            "working_year": first.get("year", "") if already_exists else pd.NA,
            "http_status": pd.NA,
            "error": pd.NA,
            "bytes": pd.NA,
        })
    df = pd.DataFrame(rows)
    return load_previous_download_status_if_available(df)


def load_previous_download_status_if_available(df):
    if df.empty or not DOWNLOAD_MANIFEST_JSONL.exists():
        return df
    try:
        prev = read_manifest(DOWNLOAD_MANIFEST_JSONL)
    except Exception as e:
        print(f"Could not load previous download manifest for status merge: {e}")
        return df
    key_cols = [c for c in ["logicDocId", "idProcedure"] if c in df.columns and c in prev.columns]
    if not key_cols:
        key_cols = [c for c in ["docId", "procedureId"] if c in df.columns and c in prev.columns]
    status_cols = [c for c in [
        "download_success", "download_status", "http_status", "error", "bytes", "working_url",
        "working_language", "working_format", "working_year", "final_local_path",
        "attempted_urls", "attempted_http_statuses", "attempt_count"
    ] if c in prev.columns]
    if not key_cols or not status_cols:
        return df
    prev_small = prev[key_cols + status_cols].drop_duplicates(subset=key_cols, keep="last")
    out = df.merge(prev_small, on=key_cols, how="left", suffixes=("", "__previous"))
    for col in status_cols:
        prev_col = f"{col}__previous"
        if prev_col in out.columns:
            has_prev = out[prev_col].notna() & out[prev_col].astype(str).ne("")
            out.loc[has_prev, col] = out.loc[has_prev, prev_col]
            out = out.drop(columns=[prev_col])
    return out


df_download_manifest = build_download_manifest(df_relevant_docs)
df_download_manifest = df_download_manifest.drop(columns=[c for c in DROP_HUGE_DOWNLOAD_COLUMNS + ["document_dedup_key"] if c in df_download_manifest.columns], errors="ignore")
def _document_identity_frame(df):
    key_cols = [c for c in ["logicDocId", "docId", "procedureId", "idProcedure", "affId"] if c in df.columns]
    return df[key_cols].astype(str).replace({"nan": "", "<NA>": ""}) if key_cols else pd.DataFrame(index=df.index)

if not df_relevant_docs.empty:
    rel_key_cols = [c for c in ["logicDocId", "docId", "procedureId", "idProcedure", "affId"] if c in df_relevant_docs.columns and c in df_download_manifest.columns]
    if rel_key_cols:
        rel_keys = df_relevant_docs[rel_key_cols].astype(str).replace({"nan": "", "<NA>": ""}).drop_duplicates()
        dl_keys = df_download_manifest[rel_key_cols].astype(str).replace({"nan": "", "<NA>": ""}).drop_duplicates()
        missing_targets = rel_keys.merge(dl_keys, on=rel_key_cols, how="left", indicator=True)
        missing_targets = missing_targets[missing_targets["_merge"].eq("left_only")].drop(columns=["_merge"])
        print("Relevant document rows targeted for download:", f"{len(df_download_manifest):,} / {len(df_relevant_docs):,}")
        print("Relevant document keys missing from download manifest:", f"{len(missing_targets):,}")
        if len(missing_targets):
            display(missing_targets.head(20))
    if "normalized_docTypeCode" in df_relevant_docs.columns:
        print("Relevant document types targeted for download, top 50:")
        display(df_relevant_docs["normalized_docTypeCode"].fillna("UNKNOWN").value_counts().head(50).to_frame("relevant_download_targets"))

print("Download manifest rows:", len(df_download_manifest))
if not df_download_manifest.empty:
    print("Rows with first candidate URL:", df_download_manifest["download_url"].fillna("").ne("").sum())
    print("Rows already existing:", df_download_manifest["file_exists_before_download"].fillna(False).astype(bool).sum())
    display(df_download_manifest[[
        c for c in ["internal_key", "procedure_key", "publishedId", "logicDocId", "normalized_docTypeCode", "docDate", "selected_language", "selected_format", "selected_year", "download_url", "intended_local_path", "existing_local_path", "download_status"]
        if c in df_download_manifest.columns
    ]].head(20))

# ---------------------------------------------------------------------
# 6) Optional downloader for relevant documents only
# ---------------------------------------------------------------------

SUCCESS_STATUSES = {"downloaded", "already_exists"}
FAILED_OR_RETRYABLE_STATUSES = {
    "pending", "failed_http", "failed_exception", "failed_missing_url", "failed_missing_target_path", "failed_empty_content",
}


def candidate_records_from_row(row):
    """Generate candidate download records in memory.

    Candidate URL lists are rebuilt from metadata instead of being stored in the manifest
    columns in the manifest. The candidates are rebuilt from metadata when needed.
    """
    records = build_candidate_download_records(row)
    if records:
        return records
    if clean_str(row.get("download_url")):
        return [{
            "url": row.get("download_url"),
            "language": row.get("selected_language"),
            "format": row.get("selected_format"),
            "year": row.get("selected_year"),
            "local_path": first_nonempty(row.get("final_local_path"), row.get("existing_local_path"), row.get("intended_local_path")),
        }]
    return []


def infer_tabs_document_url(file_url, row=None):
    try:
        parsed = urlparse(str(file_url))
        parts = parsed.path.strip("/").split("/")
        if len(parts) < 4:
            return "https://infocuria.curia.europa.eu/"
        filename = parts[-1]
        logic_fmt = filename.replace(".", "-")
        doc_type = first_nonempty((row or {}).get("normalized_docTypeCode"), (row or {}).get("docTypeCode"))
        routing = first_nonempty((row or {}).get("idProcedure"), (row or {}).get("procedureId"))
        if routing and doc_type:
            routing_pretty = routing.replace("/", "-")
            return f"https://infocuria.curia.europa.eu/tabs/document/{routing_pretty}/{doc_type}/{logic_fmt}"
    except Exception:
        pass
    return "https://infocuria.curia.europa.eu/"


def get_download_session():
    session = requests.Session()
    session.headers.update(HEADERS)
    return session


def polite_sleep(base=None):
    base = DOWNLOAD_SLEEP_SECONDS if base is None else base
    delay = float(base) + random.uniform(0, float(DOWNLOAD_SLEEP_JITTER_SECONDS))
    if delay > 0:
        time.sleep(delay)


def warmup_session_for_url(session, file_url, row):
    if not SESSION_WARMUP:
        return None
    referrer = infer_tabs_document_url(file_url, row=row)
    try:
        session.get(referrer, timeout=REQUEST_TIMEOUT_SECONDS, headers={**HEADERS, "Referer": "https://infocuria.curia.europa.eu/"}, allow_redirects=True)
        time.sleep(float(SESSION_WARMUP_SLEEP_SECONDS))
        return referrer
    except Exception:
        return "https://infocuria.curia.europa.eu/"


def download_one_file(row, overwrite=False, session=None):
    records = candidate_records_from_row(row)
    if not records:
        return {
            "download_success": False,
            "download_status": "failed_missing_url",
            "http_status": pd.NA,
            "error": "Missing candidate_download_records/download_url",
            "bytes": pd.NA,
            "working_url": pd.NA,
            "working_language": pd.NA,
            "working_format": pd.NA,
            "working_year": pd.NA,
            "final_local_path": pd.NA,
            "attempted_urls": json.dumps([], ensure_ascii=False),
            "attempted_http_statuses": json.dumps([], ensure_ascii=False),
            "attempt_count": 0,
        }

    session = session or get_download_session()
    attempted_urls = []
    attempted_statuses = []
    last_error = ""
    last_status = pd.NA
    last_length = 0

    for record in records:
        file_url = clean_str(record.get("url"))
        target_path = Path(clean_str(record.get("local_path"))) if clean_str(record.get("local_path")) else None
        selected_format = clean_str(record.get("format")).upper()

        if not file_url or target_path is None:
            continue

        if target_path.exists() and not overwrite:
            return {
                "download_success": True,
                "download_status": "already_exists",
                "http_status": pd.NA,
                "error": pd.NA,
                "bytes": target_path.stat().st_size,
                "working_url": file_url,
                "working_language": record.get("language"),
                "working_format": selected_format,
                "working_year": record.get("year"),
                "final_local_path": str(target_path),
                "attempted_urls": json.dumps(attempted_urls, ensure_ascii=False),
                "attempted_http_statuses": json.dumps(attempted_statuses, ensure_ascii=False),
                "attempt_count": len(attempted_urls),
            }

        referrer = warmup_session_for_url(session, file_url, row)

        for retry_no in range(1, int(RETRIES_PER_URL) + 1):
            attempted_urls.append(file_url)
            try:
                response = session.get(
                    file_url,
                    timeout=REQUEST_TIMEOUT_SECONDS,
                    headers={**HEADERS, "Referer": referrer or "https://infocuria.curia.europa.eu/"},
                    allow_redirects=True,
                )
                status_code = response.status_code
                content = response.content or b""
                last_status = status_code
                last_length = len(content)
                attempted_statuses.append(status_code)

                if status_code == 200 and content:
                    target_path.parent.mkdir(parents=True, exist_ok=True)
                    if selected_format == "HTML" or file_url.lower().endswith(".html"):
                        target_path.write_text(response.text, encoding="utf-8")
                    else:
                        target_path.write_bytes(content)
                    return {
                        "download_success": True,
                        "download_status": "downloaded",
                        "http_status": status_code,
                        "error": pd.NA,
                        "bytes": target_path.stat().st_size,
                        "working_url": file_url,
                        "working_language": record.get("language"),
                        "working_format": selected_format,
                        "working_year": record.get("year"),
                        "final_local_path": str(target_path),
                        "attempted_urls": json.dumps(attempted_urls, ensure_ascii=False),
                        "attempted_http_statuses": json.dumps(attempted_statuses, ensure_ascii=False),
                        "attempt_count": len(attempted_urls),
                    }

                last_error = f"HTTP {status_code}; length={len(content)}"
                if status_code in {403, 429, 500, 502, 503, 504} and retry_no < int(RETRIES_PER_URL):
                    time.sleep(float(RETRY_BACKOFF_SECONDS) * retry_no + random.uniform(0, 1.5))
                    continue
                # 404: try the next URL candidate immediately.
                break
            except Exception as e:
                last_error = repr(e)
                attempted_statuses.append("EXCEPTION")
                if retry_no < int(RETRIES_PER_URL):
                    time.sleep(float(RETRY_BACKOFF_SECONDS) * retry_no + random.uniform(0, 1.5))
                    continue
                break
        polite_sleep()

    return {
        "download_success": False,
        "download_status": "failed_http" if attempted_urls else "failed_missing_url",
        "http_status": last_status,
        "error": f"{last_error}; last_length={last_length}",
        "bytes": last_length,
        "working_url": pd.NA,
        "working_language": pd.NA,
        "working_format": pd.NA,
        "working_year": pd.NA,
        "final_local_path": pd.NA,
        "attempted_urls": json.dumps(attempted_urls, ensure_ascii=False),
        "attempted_http_statuses": json.dumps(attempted_statuses, ensure_ascii=False),
        "attempt_count": len(attempted_urls),
    }


def run_downloads(df_download, limit=None, overwrite=False, save_every=100):
    df = df_download.copy()
    if df.empty:
        print("Download manifest is empty.")
        return df

    # If a final/specified local path exists, mark it as already present.
    # Use NA-safe helpers: pandas pd.NA cannot be used with `or`/`if`.
    existing_now = df.apply(
        lambda r: path_exists_safe(r.get("final_local_path"), r.get("existing_local_path"), r.get("intended_local_path")),
        axis=1,
    )
    if not overwrite:
        df.loc[existing_now, "download_success"] = True
        df.loc[existing_now, "download_status"] = "already_exists"

    status = df["download_status"].fillna("pending").astype(str)
    success_bool = df.get("download_success", pd.Series(False, index=df.index)).astype(str).str.lower().isin(["true", "1", "yes"])
    has_candidates = df["download_url"].fillna("").astype(str).str.strip().ne("")

    needs_attempt = (
        has_candidates
        & (overwrite | ~existing_now)
        & (
            overwrite
            | status.isin(FAILED_OR_RETRYABLE_STATUSES)
            | status.str.startswith("failed")
            | (~success_bool & ~status.isin(SUCCESS_STATUSES))
        )
        & ~status.isin(SUCCESS_STATUSES)
    )

    pending_indices = df.index[needs_attempt].tolist()
    if limit is not None:
        pending_indices = pending_indices[:int(limit)]

    print(f"Download attempts planned: {len(pending_indices):,}")
    session = get_download_session()
    for n, idx in enumerate(tqdm(pending_indices, desc="Downloading relevant InfoCuria documents"), start=1):
        result = download_one_file(df.loc[idx].to_dict(), overwrite=overwrite, session=session)
        for k, v in result.items():
            df.at[idx, k] = v
        if save_every and n % int(save_every) == 0:
            save_jsonl(df.drop(columns=[c for c in DROP_HUGE_DOWNLOAD_COLUMNS + ["document_dedup_key"] if c in df.columns], errors="ignore"), DOWNLOAD_MANIFEST_JSONL)

    df = df.drop(columns=[c for c in DROP_HUGE_DOWNLOAD_COLUMNS + ["document_dedup_key"] if c in df.columns], errors="ignore")
    save_jsonl(df, DOWNLOAD_MANIFEST_JSONL)
    return df


if RUN_DOWNLOADS:
    df_download_manifest = run_downloads(
        df_download_manifest,
        limit=DOWNLOAD_LIMIT,
        overwrite=OVERWRITE_EXISTING_FILES,
        save_every=DOWNLOAD_SAVE_EVERY,
    )
else:
    print("RUN_DOWNLOADS=False. No documents downloaded.")
    print("The download manifest contains selected/attempted URLs and previous/current status where available.")


## 7. Save the Court tables

Save three XLSX workbooks: enriched cases, documents, and download targets. The third is the explicit handoff to text processing. Keep unsuccessful downloads visible in that table so missing text files can be explained.


In [ ]:
# ---------------------------------------------------------------------
# 9) Save requested manifests
# ---------------------------------------------------------------------


# ---------------------------------------------------------------------
# ---------------------------------------------------------------------

DOWNLOAD_SUCCESS_STATUSES = {"downloaded", "already_exists"}

def _truthy_download_success(s):
    return s.astype(str).str.lower().isin(["true", "1", "yes"])


def _download_path_exists_row(row):
    return path_exists_safe(
        row.get("final_local_path"),
        row.get("existing_local_path"),
        row.get("intended_local_path"),
    )


def attach_download_status_to_documents(df_docs: pd.DataFrame, df_downloads: pd.DataFrame) -> pd.DataFrame:
    docs = df_docs.copy()
    if docs.empty:
        return docs

    default_cols = {
        "download_target_flag": False,
        "download_success": False,
        "download_status": "not_targeted_for_download",
        "file_present_on_disk": False,
        "download_url": "",
        "working_url": "",
        "final_local_path": "",
        "intended_local_path": "",
        "existing_local_path": "",
        "attempted_urls": "[]",
        "attempt_count": 0,
    }
    for col, val in default_cols.items():
        if col not in docs.columns:
            docs[col] = val

    if df_downloads is None or df_downloads.empty:
        return docs

    dl = df_downloads.copy()
    if "file_present_on_disk" not in dl.columns:
        dl["file_present_on_disk"] = dl.apply(_download_path_exists_row, axis=1)
    else:
        dl["file_present_on_disk"] = dl["file_present_on_disk"].astype(str).str.lower().isin(["true", "1", "yes"]) | dl.apply(_download_path_exists_row, axis=1)

    if "download_success" not in dl.columns:
        dl["download_success"] = False
    dl["download_success"] = _truthy_download_success(dl["download_success"]) | dl["file_present_on_disk"] | dl.get("download_status", pd.Series("", index=dl.index)).astype(str).isin(DOWNLOAD_SUCCESS_STATUSES)

    key_cols = [c for c in ["logicDocId", "docId", "procedureId", "idProcedure", "affId"] if c in docs.columns and c in dl.columns]
    if not key_cols:
        print("Warning: could not attach download status to document metadata; no shared document key columns.")
        return docs

    dl_cols = key_cols + [c for c in [
        "download_success", "download_status", "file_present_on_disk", "download_url", "working_url",
        "final_local_path", "intended_local_path", "existing_local_path", "attempted_urls", "attempt_count",
    ] if c in dl.columns]
    dl_small = dl[dl_cols].drop_duplicates(subset=key_cols, keep="last")
    merged = docs.merge(dl_small, on=key_cols, how="left", suffixes=("", "__download"))

    for col, default_val in default_cols.items():
        dl_col = f"{col}__download"
        if dl_col in merged.columns:
            has = merged[dl_col].notna() & merged[dl_col].astype(str).ne("")
            merged.loc[has, col] = merged.loc[has, dl_col]
            merged = merged.drop(columns=[dl_col])

    merged["download_target_flag"] = merged["download_status"].fillna("not_targeted_for_download").astype(str).ne("not_targeted_for_download")
    merged["download_success"] = _truthy_download_success(merged["download_success"])
    merged["file_present_on_disk"] = merged["file_present_on_disk"].astype(str).str.lower().isin(["true", "1", "yes"]) | merged.apply(_download_path_exists_row, axis=1)
    return merged


def _join_ids(values, max_items=5000):
    return unique_join(values, sep="|", max_items=max_items)


def add_case_download_rollup_by_procedure(
    df_cases_manifest: pd.DataFrame,
    df_docs_with_downloads: pd.DataFrame,
) -> pd.DataFrame:
    """
    Roll document/download status up to the case manifest.

    procedureId is the canonical grouping and merge key. affId is used
    only as a fallback when procedureId is unavailable.
    """
    out = df_cases_manifest.copy()

    if out.empty or df_docs_with_downloads.empty:
        return out

    if (
        "procedureId" in out.columns
        and "procedureId" in df_docs_with_downloads.columns
        and out["procedureId"].fillna("").astype(str).str.strip().ne("").any()
        and df_docs_with_downloads["procedureId"]
            .fillna("")
            .astype(str)
            .str.strip()
            .ne("")
            .any()
    ):
        group_col = "procedureId"

    elif (
        "affId" in out.columns
        and "affId" in df_docs_with_downloads.columns
    ):
        group_col = "affId"
        print(
            "Warning: procedureId unavailable for download roll-up; "
            "falling back to affId."
        )

    else:
        print(
            "Warning: could not roll up downloads; no shared "
            "procedureId or affId column."
        )
        return out

    docs = df_docs_with_downloads.copy()

    docs["relevance_flag"] = (
        bool_series(docs["relevance_flag"])
        if "relevance_flag" in docs.columns
        else False
    )

    docs["download_success"] = (
        _truthy_download_success(docs["download_success"])
        if "download_success" in docs.columns
        else False
    )

    docs["download_target_flag"] = (
        docs["download_target_flag"]
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes"])
        if "download_target_flag" in docs.columns
        else False
    )

    docs["file_present_on_disk"] = (
        docs["file_present_on_disk"]
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes"])
        if "file_present_on_disk" in docs.columns
        else False
    )

    docs["downloaded_or_present"] = (
        docs["download_success"]
        | docs["file_present_on_disk"]
    )

    document_id_cols = [
        c
        for c in ["logicDocId", "docId", "idPublished"]
        if c in docs.columns
    ]

    if document_id_cols:
        docs["document_id_for_status"] = (
            docs[document_id_cols]
            .astype(str)
            .replace({"nan": "", "<NA>": ""})
            .agg(lambda values: first_nonempty(*values), axis=1)
        )
    else:
        docs["document_id_for_status"] = ""

    rel_docs = docs[docs["relevance_flag"]].copy()
    rel_not_downloaded = rel_docs[
        ~rel_docs["downloaded_or_present"]
    ].copy()
    rel_downloaded = rel_docs[
        rel_docs["downloaded_or_present"]
    ].copy()

    agg = (
        docs
        .groupby(group_col, dropna=False)
        .agg(
            all_document_count=("document_id_for_status", "count"),
            all_download_target_document_count=(
                "download_target_flag",
                "sum",
            ),
            all_downloaded_or_present_document_count=(
                "downloaded_or_present",
                "sum",
            ),
            all_document_download_statuses=(
                "download_status",
                unique_join,
            ),
        )
        .reset_index()
    )

    rel_count_col = "relevant_document_count_by_procedure"

    rel_agg = (
        rel_docs
        .groupby(group_col, dropna=False)
        .agg(
            relevant_document_count_by_procedure=(
                "document_id_for_status",
                "count",
            ),
            relevant_download_target_document_count=(
                "download_target_flag",
                "sum",
            ),
            relevant_downloaded_or_present_document_count=(
                "downloaded_or_present",
                "sum",
            ),
            relevant_document_download_statuses=(
                "download_status",
                unique_join,
            ),
            relevant_document_ids=(
                "document_id_for_status",
                _join_ids,
            ),
        )
        .reset_index()
    )

    rel_down_ids = (
        rel_downloaded
        .groupby(group_col, dropna=False)["document_id_for_status"]
        .apply(_join_ids)
        .reset_index(name="relevant_downloaded_document_ids")
    )

    rel_not_down_ids = (
        rel_not_downloaded
        .groupby(group_col, dropna=False)["document_id_for_status"]
        .apply(_join_ids)
        .reset_index(name="relevant_not_downloaded_document_ids")
    )

    roll = agg.merge(rel_agg, on=group_col, how="left")
    roll = roll.merge(rel_down_ids, on=group_col, how="left")
    roll = roll.merge(rel_not_down_ids, on=group_col, how="left")

    numeric_cols = [
        rel_count_col,
        "relevant_download_target_document_count",
        "relevant_downloaded_or_present_document_count",
    ]

    for column in numeric_cols:
        if column not in roll.columns:
            roll[column] = 0

        roll[column] = (
            pd.to_numeric(roll[column], errors="coerce")
            .fillna(0)
            .astype(int)
        )

    text_cols = [
        "relevant_document_download_statuses",
        "relevant_document_ids",
        "relevant_downloaded_document_ids",
        "relevant_not_downloaded_document_ids",
    ]

    for column in text_cols:
        if column not in roll.columns:
            roll[column] = ""

        roll[column] = roll[column].fillna("")

    roll["case_has_any_document"] = (
        roll["all_document_count"].gt(0)
    )

    roll["case_has_relevant_document"] = (
        roll[rel_count_col].gt(0)
    )

    roll["case_has_downloaded_relevant_document"] = (
        roll["relevant_downloaded_or_present_document_count"].gt(0)
    )

    roll["case_all_relevant_documents_downloaded"] = (
        roll["case_has_relevant_document"]
        & roll["relevant_downloaded_or_present_document_count"].eq(
            roll[rel_count_col]
        )
    )

    roll["case_any_document_downloaded_or_present"] = (
        roll["all_downloaded_or_present_document_count"].gt(0)
    )

    roll["case_all_targeted_documents_downloaded"] = (
        roll["all_download_target_document_count"].gt(0)
        & roll["all_downloaded_or_present_document_count"].eq(
            roll["all_download_target_document_count"]
        )
    )

    replace_cols = [
        c
        for c in roll.columns
        if c != group_col
    ]

    out = out.drop(
        columns=[
            c
            for c in replace_cols
            if c in out.columns
        ],
        errors="ignore",
    )

    out = out.merge(
        roll,
        on=group_col,
        how="left",
    )

    out["case_relevance_flag"] = (
        out["case_has_relevant_document"]
        .fillna(False)
        .astype(bool)
    )

    out["relevant_document_count"] = (
        pd.to_numeric(
            out[rel_count_col],
            errors="coerce",
        )
        .fillna(0)
        .astype(int)
    )

    return out




# Attach document-level download status and roll it up to the procedure manifest.
df_doc_meta = attach_download_status_to_documents(df_doc_meta, df_download_manifest)
df_case_metadata_manifest = add_case_download_rollup_by_procedure(df_case_metadata_manifest, df_doc_meta)
df_document_metadata_manifest = df_doc_meta.copy()

# Keep the final download manifest compact and make the raw-file handoff explicit.
def repo_path(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "<na>"}:
        return ""
    path = Path(text)
    try:
        return path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
    except Exception:
        return path.as_posix()


def add_raw_file_handoff(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    raw_paths = []
    raw_names = []
    raw_formats = []
    success = []
    for _, row in out.iterrows():
        chosen = first_nonempty(row.get("final_local_path"), row.get("existing_local_path"), row.get("intended_local_path"))
        path = Path(chosen) if chosen else None
        exists = bool(path and path.exists() and path.is_file())
        raw_paths.append(repo_path(path) if exists else "")
        raw_names.append(path.name if exists else "")
        raw_formats.append(path.suffix.lower().lstrip(".") if exists else clean_str(row.get("working_format")).lower())
        success.append(bool(exists or str(row.get("download_success", "")).lower() in {"true", "1", "yes"}))
    out["success"] = success
    out["raw_file_name"] = raw_names
    out["raw_file_path"] = raw_paths
    out["file_format"] = raw_formats
    return out


df_download_manifest = add_raw_file_handoff(df_download_manifest)

for frame_name in ["df_case_metadata_manifest", "df_document_metadata_manifest", "df_download_manifest"]:
    frame = globals()[frame_name]
    frame = frame.drop(columns=[c for c in DROP_NON_ENGLISH_METADATA_COLUMNS if c in frame.columns], errors="ignore")
    if frame_name in {"df_document_metadata_manifest", "df_download_manifest"}:
        frame = frame.drop(columns=[c for c in ["document_dedup_key"] + DROP_HUGE_DOWNLOAD_COLUMNS if c in frame.columns], errors="ignore")
    globals()[frame_name] = frame

save_table(df_case_metadata_manifest, CASE_METADATA_XLSX, "cases")
save_table(df_document_metadata_manifest, DOCUMENT_METADATA_XLSX, "documents")
save_table(df_download_manifest, DOWNLOAD_MANIFEST_XLSX, "downloads")

print("Saved case metadata:", CASE_METADATA_XLSX)
print("Saved document metadata:", DOCUMENT_METADATA_XLSX)
print("Saved download manifest:", DOWNLOAD_MANIFEST_XLSX)


## 8. Check the handoff

Summarize case, document, and download counts; duplicate non-empty procedure identifiers; relevant documents; and successful raw files. Review unmatched or unsuccessful rows before using the collection downstream.


In [ ]:
print(f"Case procedures: {len(df_case_metadata_manifest):,}")
print(f"Documents: {len(df_document_metadata_manifest):,}")
print(f"Download targets: {len(df_download_manifest):,}")

if "procedureId" in df_case_metadata_manifest.columns:
    nonempty = df_case_metadata_manifest["procedureId"].fillna("").astype(str).str.strip().ne("")
    print("Duplicate non-empty procedureId rows:", int(df_case_metadata_manifest.loc[nonempty, "procedureId"].duplicated().sum()))

if "relevance_flag" in df_document_metadata_manifest.columns:
    print("Relevant documents:", int(bool_series(df_document_metadata_manifest["relevance_flag"]).sum()))

if "download_status" in df_download_manifest.columns:
    display(df_download_manifest["download_status"].fillna("missing").value_counts(dropna=False).rename_axis("download_status").reset_index(name="rows"))

print("Rows with a downloaded raw file:", int(df_download_manifest["raw_file_path"].fillna("").astype(str).str.strip().ne("").sum()))
